In [ ]:
import h5py
import scipy.io
import numpy as np
import os
from google.colab import drive
src_path    = '/content/drive/MyDrive/THINGS_MUA_trials_monkeyF (1).mat'
map_path    = '/content/drive/MyDrive/1024chns_mapping_20220105 (3)Monkey-F.mat'
local_cache = '/content/monkey_full_cleaned_normalized.h5'
print("--- Step 1: Loading Full Mapping (1024 Channels) ---")
mapping = scipy.io.loadmat(map_path)['mapping'].flatten().astype(int) - 1
good_mapped_indices = np.arange(1024)
source_indices      = mapping[good_mapped_indices]
print("--- Step 2: High-Speed Load from Drive (~62GB) ---")
with h5py.File(src_path, 'r') as f:
    data_raw = f['ALLMUA'][:]
print("--- Step 3: Reordering and Converting to Float32 ---")
data_raw = data_raw[:, :, source_indices].astype(np.float32)
T_total, N_trials, N_units = data_raw.shape
print(f"    Raw shape  →  T={T_total} ms | Trials={N_trials} | Channels={N_units}")
assert T_total  == 300,  f"Expected 300 time bins, got {T_total}"
assert N_units  == 1024, f"Expected 1024 channels after mapping, got {N_units}"
print("\n--- Step 4: Computing SNR and SNRmax on RAW MUA (pre z-score) ---")
BASELINE_START = 0
BASELINE_END   = 100
SMOOTH_WIN     = 25
SNR_THRESHOLD  = 0.0
MIN_ELECTRODES = 100
baseline_raw = data_raw[BASELINE_START:BASELINE_END, :, :]
baseline_trial_mean = baseline_raw.mean(axis=0)
baseline_trial_std  = baseline_raw.std(axis=0)
mean_baseline = baseline_trial_mean.mean(axis=0)
sd_baseline   = baseline_trial_std.mean(axis=0)
sd_baseline = np.where(sd_baseline < 1e-9, 1e-9, sd_baseline)
def smooth_psth(psth, win=SMOOTH_WIN):
    """
    Moving-average smooth along time axis (axis=0), per channel.
    Uses float64 internally for numerical stability.
    psth shape: (T, N_channels)
    """
    kernel = np.ones(win, dtype=np.float64) / win
    out    = np.empty_like(psth)
    for ch in range(psth.shape[1]):
        out[:, ch] = np.convolve(
            psth[:, ch].astype(np.float64), kernel, mode='same'
        ).astype(psth.dtype)
    return out
psth_all      = data_raw.mean(axis=1)
print("    Smoothing all-image PSTH...")
psth_smoothed = smooth_psth(psth_all)
peak_all = psth_smoothed[BASELINE_END:, :].max(axis=0)
SNR = (peak_all - mean_baseline) / sd_baseline
print("    Identifying preferred image per channel for SNRmax...")
response_raw       = data_raw[BASELINE_END:, :, :]
mean_resp_per_trial = response_raw.mean(axis=0)
preferred_trial_idx = mean_resp_per_trial.argmax(axis=0)
preferred_psth = np.stack(
    [data_raw[:, preferred_trial_idx[ch], ch] for ch in range(N_units)],
    axis=1
).astype(np.float32)
print("    Smoothing preferred-image PSTH...")
preferred_smoothed = smooth_psth(preferred_psth)
peak_preferred     = preferred_smoothed[BASELINE_END:, :].max(axis=0)
SNRmax = (peak_preferred - mean_baseline) / sd_baseline
ROI_SLICES = {
    'V1': slice(0,   512),
    'IT': slice(512, 832),
    'V4': slice(832, 1024),
}
print("\n    SNR summary per ROI (raw MUA, before thresholding):")
print(f"    {'ROI':<6} {'N_ch':>5} {'SNR_mean':>10} {'SNR_med':>9} "
      f"{'SNR_min':>8} {'SNR_max_val':>12} {'SNRmax_mean':>13} {'N_below_thr':>12}")
print("    " + "-" * 80)
for roi, sl in ROI_SLICES.items():
    s    = SNR[sl]
    sm   = SNRmax[sl]
    n_bad = int((s < SNR_THRESHOLD).sum())
    print(f"    {roi:<6} {len(s):>5} {s.mean():>10.3f} {np.median(s):>9.3f} "
          f"{s.min():>8.3f} {s.max():>12.3f} {sm.mean():>13.3f} {n_bad:>12}")
print(f"\n--- Step 5: Applying SNR Threshold (SNR >= {SNR_THRESHOLD}, min {MIN_ELECTRODES} per ROI) ---")
good_mask = np.zeros(N_units, dtype=bool)
for roi, sl in ROI_SLICES.items():
    roi_snr        = SNR[sl]
    roi_abs_indices = np.arange(sl.start, sl.stop)
    passing = roi_snr >= SNR_THRESHOLD
    n_passing = int(passing.sum())
    if n_passing >= MIN_ELECTRODES:
        selected = passing
        method   = f"SNR threshold ({n_passing} passed)"
    else:
        top_indices = np.argsort(roi_snr)[-MIN_ELECTRODES:]
        selected    = np.zeros(len(roi_snr), dtype=bool)
        selected[top_indices] = True
        method = (f"fallback top-{MIN_ELECTRODES} by SNR "
                  f"(only {n_passing} passed threshold)")
    good_mask[roi_abs_indices] = selected
    print(f"    {roi}: {method}")
n_good      = int(good_mask.sum())
n_bad_total = int((~good_mask).sum())
print(f"\n    Total channels   : {N_units}")
print(f"    Channels passing : {n_good}")
print(f"    Channels removed : {n_bad_total}")
print("\n    Survival per ROI after thresholding:")
print(f"    {'ROI':<6} {'Kept':>6} {'Removed':>8} {'Total':>7} {'%Kept':>8}")
print("    " + "-" * 38)
for roi, sl in ROI_SLICES.items():
    kept    = int(good_mask[sl].sum())
    removed = int((~good_mask[sl]).sum())
    total   = sl.stop - sl.start
    pct     = 100.0 * kept / total
    print(f"    {roi:<6} {kept:>6} {removed:>8} {total:>7} {pct:>7.1f}%")
data_filtered = data_raw[:, :, good_mask]
print(f"\n    Filtered data shape (raw): {data_filtered.shape}")
del data_raw, psth_all, psth_smoothed, preferred_psth, preferred_smoothed
del baseline_raw, response_raw, mean_resp_per_trial
snr_save_path = '/content/monkeyF_SNR_per_channel.npz'
np.savez(snr_save_path,
         SNR=SNR,
         SNRmax=SNRmax,
         good_mask=good_mask,
         mean_baseline=mean_baseline,
         sd_baseline=sd_baseline,
         peak_all=peak_all,
         peak_preferred=peak_preferred)
print(f"    SNR arrays saved → {snr_save_path}")
print("\n--- Step 6: In-Place Z-Scoring (filtered channels only) ---")
mu    = data_filtered.mean(axis=(0, 1), keepdims=True)
sigma = data_filtered.std(axis=(0, 1),  keepdims=True)
data_filtered -= mu
data_filtered /= (sigma + 1e-9)
print(f"    Z-scoring complete. Shape: {data_filtered.shape}")
print(f"\n--- Step 7: Caching Filtered + Normalised Data ({local_cache}) ---")
good_indices = np.where(good_mask)[0]
roi_counts = {}
for roi, sl in ROI_SLICES.items():
    in_roi          = (good_indices >= sl.start) & (good_indices < sl.stop)
    roi_counts[roi] = int(in_roi.sum())
    print(f"    {roi} → {roi_counts[roi]} channels retained in filtered array")
v1_end = roi_counts['V1']
it_end = v1_end + roi_counts['IT']
with h5py.File(local_cache, 'w') as hf:
    hf.create_dataset('data',      data=data_filtered,         compression=None)
    hf.create_dataset('good_mask', data=good_mask.astype(np.uint8))
    hf.create_dataset('SNR',       data=SNR.astype(np.float32))
    hf.create_dataset('SNRmax',    data=SNRmax.astype(np.float32))
    hf.attrs['ROI_V1_end']      = v1_end
    hf.attrs['ROI_IT_end']      = it_end
    hf.attrs['SNR_threshold']   = SNR_THRESHOLD
    hf.attrs['n_good_channels'] = n_good
    hf.attrs['ROI_slices']      = (
        f"V1: 0:{v1_end}, IT: {v1_end}:{it_end}, "
        f"V4: {it_end}:{n_good}  "
        f"(SNR >= {SNR_THRESHOLD} on raw MUA, computed pre z-score; "
        f"min {MIN_ELECTRODES} per ROI fallback applied)"
    )
print("\n=== Pipeline Complete ===")
print(f"Final filtered data shape  : {data_filtered.shape}")
print(f"Local storage used         : {os.path.getsize(local_cache) / 1e9:.2f} GB")
print(f"ROI boundaries (filtered)  : "
      f"V1=0:{v1_end}, IT={v1_end}:{it_end}, V4={it_end}:{n_good}")
print(f"SNR data saved to          : {snr_save_path}")
print(f"\nUse these in your analysis pipeline:")
print(f"    with h5py.File(CACHE_PATH, 'r') as hf:")
print(f"        clean_data = hf['data'][:].astype(np.float64)")
print(f"        v1_end     = int(hf.attrs['ROI_V1_end'])   # {v1_end}")
print(f"        it_end     = int(hf.attrs['ROI_IT_end'])   # {it_end}")
print(f"    roi_clean = {{")
print(f"        'V1': clean_data[:, :,      :v1_end],")
print(f"        'IT': clean_data[:, :, v1_end:it_end],")
print(f"        'V4': clean_data[:, :, it_end:     ],")
print(f"    }}")

--- Step 1: Loading Full Mapping (1024 Channels) ---
--- Step 2: High-Speed Load from Drive (~62GB) ---
--- Step 3: Reordering and Converting to Float32 ---
    Raw shape  →  T=300 ms | Trials=25248 | Channels=1024

--- Step 4: Computing SNR and SNRmax on RAW MUA (pre z-score) ---
    Smoothing all-image PSTH...
    Identifying preferred image per channel for SNRmax...
    Smoothing preferred-image PSTH...

    SNR summary per ROI (raw MUA, before thresholding):
    ROI     N_ch   SNR_mean   SNR_med  SNR_min  SNR_max_val   SNRmax_mean  N_below_thr
    --------------------------------------------------------------------------------
    V1       512      1.320     1.025   -0.086        5.900        12.642            3
    IT       320      0.470     0.374   -0.068        2.095        10.418            9
    V4       192      0.985     0.849    0.032        4.365        13.009            0

--- Step 5: Applying SNR Threshold (SNR >= 0.0, min 100 per ROI) ---
    V1: SNR threshold (509 pas

In [ ]:
'''
ROI_SLICES = {
    'V1': slice(0,   512),
    'IT': slice(512, 832),
    'V4': slice(832, 1024),
}
'''
import h5py
import scipy.io
import numpy as np
import os
from google.colab import drive
drive.mount('/content/drive')
src_path = '/content/drive/MyDrive/THINGS_MUA_trials_monkeyF (1).mat'
map_path = '/content/drive/MyDrive/1024chns_mapping_20220105 (3)Monkey-F.mat'
local_cache = '/content/monkey_full_cleaned_normalized.h5'
print("--- Step 1: Loading Mapping ---")
mapping = scipy.io.loadmat(map_path)['mapping'].flatten().astype(int) - 1
good_mapped_indices = np.arange(1024)
source_indices = mapping[good_mapped_indices]
print("--- Step 2: High-Speed Load from Drive (~62GB) ---")
with h5py.File(src_path, 'r') as f:
    data = f['ALLMUA'][:]
print("--- Step 3: Reordering and Converting to Float32 ---")
data = data[:, :, source_indices].astype(np.float32)
print("--- Step 4: Rigorous In-Place Z-Scoring ---")
mu = data.mean(axis=(0, 1), keepdims=True)
sigma = data.std(axis=(0, 1), keepdims=True)
data -= mu
data /= (sigma + 1e-9)
print(f"--- Step 5: Caching to Local Runtime ({local_cache}) ---")
with h5py.File(local_cache, 'w') as hf:
    hf.create_dataset('data', data=data, compression=None)
    hf.attrs['ROI_slices'] = "V1: 0-512, V4: 512-832, IT: 832-1024"
print("\nPipeline Complete!")
print(f"Final RAM data shape: {data.shape}")
print(f"Local storage used: {os.path.getsize(local_cache) / 1e9:.2f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Step 1: Loading Mapping ---
--- Step 2: High-Speed Load from Drive (~62GB) ---
--- Step 3: Reordering and Converting to Float32 ---
--- Step 4: Rigorous In-Place Z-Scoring ---
--- Step 5: Caching to Local Runtime (/content/monkey_full_cleaned_normalized.h5) ---

Pipeline Complete!
Final RAM data shape: (300, 25248, 1024)
Local storage used: 31.02 GB


In [ ]:
import h5py
import numpy as np
file_path = '/content/drive/MyDrive/THINGS_MUA_trials_monkeyF (1).mat'
with h5py.File(file_path, 'r') as f:
    metadata = np.array(f['ALLMAT'])
    trial_nums = metadata[0, :]
    train_ids = metadata[1, :]
    test_ids = metadata[2, :]
    counts = metadata[4, :]
    print(f"--- Dataset Summary ---")
    print(f"Total Stimulus Presentations: {metadata.shape[1]}")
    print("\nFirst 10 Trials Preview:")
    print(f"{'Trial #':<10} | {'Type':<8} | {'Stimulus ID':<12} | {'Seq Position'}")
    print("-" * 50)
    for i in range(10):
        is_train = train_ids[i] != 0
        stim_id = train_ids[i] if is_train else test_ids[i]
        trial_type = "TRAIN" if is_train else "TEST"
        print(f"{int(trial_nums[i]):<10} | {trial_type:<8} | {int(stim_id):<12} | {int(counts[i])}")

--- Dataset Summary ---
Total Stimulus Presentations: 25248

First 10 Trials Preview:
Trial #    | Type     | Stimulus ID  | Seq Position
--------------------------------------------------
1          | TRAIN    | 16504        | 1
2          | TRAIN    | 16470        | 2
3          | TRAIN    | 15094        | 3
4          | TRAIN    | 2514         | 4
5          | TRAIN    | 4860         | 1
6          | TEST     | 27           | 2
7          | TEST     | 5            | 3
8          | TRAIN    | 7629         | 4
9          | TEST     | 64           | 1
10         | TRAIN    | 16848        | 2


In [ ]:
import h5py
import numpy as np
import pandas as pd
import os
mat_mapping_path = '/content/drive/MyDrive/things_imgs (2)Monkey-F.mat'
neural_data_path = '/content/drive/MyDrive/THINGS_MUA_trials_monkeyF (1).mat'
extract_dir = '/content/drive/MyDrive/THINGS_images_unzipped'
def load_instance_paths(f, group_name, field_name):
    """
    Dereferences the specific field (like 'things_path') from
    the MATLAB v7.3 group.
    """
    dataset = f[group_name][field_name]
    refs = dataset[:].flatten()
    strings = []
    print(f"Dereferencing {len(refs)} items from {group_name}/{field_name}...")
    for ref in refs:
        obj = f[ref]
        s = "".join(chr(c) for c in np.array(obj).flatten()).replace('\\', '/')
        strings.append(s)
    return strings
with h5py.File(mat_mapping_path, 'r') as f:
    train_instances = load_instance_paths(f, 'train_imgs', 'things_path')
    test_instances = load_instance_paths(f, 'test_imgs', 'things_path')
print("Aligning 25,248 trials to instance filenames...")
with h5py.File(neural_data_path, 'r') as f:
    all_mat = f['ALLMAT'][:]
    train_idx = all_mat[1, :].astype(int)
    test_idx = all_mat[2, :].astype(int)
final_paths = []
final_concepts = []
for i in range(25248):
    if train_idx[i] > 0:
        rel_path = train_instances[train_idx[i] - 1]
        is_train = True
    else:
        rel_path = test_instances[test_idx[i] - 1]
        is_train = False
    final_paths.append(os.path.join(extract_dir, rel_path))
    final_concepts.append(rel_path.split('/')[0])
master_metadata = pd.DataFrame({
    'trial_idx': np.arange(25248),
    'concept': final_concepts,
    'relative_path': [p.replace(extract_dir + '/', '') for p in final_paths],
    'full_path': final_paths,
    'is_train': train_idx > 0
})
print(f"\nSuccess! All {len(master_metadata)} trials are matched to instances.")
print(f"Sample Instance: {master_metadata.iloc[0]['relative_path']}")

Dereferencing 22248 items from train_imgs/things_path...
Dereferencing 100 items from test_imgs/things_path...
Aligning 25,248 trials to instance filenames...

Success! All 25248 trials are matched to instances.
Sample Instance: scanner/scanner_04s.jpg


In [ ]:
import os, warnings, time
warnings.filterwarnings("ignore", category=RuntimeWarning)
import numpy as np, pandas as pd, h5py, torch
import torchvision.models as models, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from numpy.linalg import slogdet
from scipy.linalg import eigh
from scipy.stats import spearmanr, normaltest
from sklearn.decomposition import PCA
import matplotlib, matplotlib.pyplot as plt
from matplotlib.colors import Normalize
matplotlib.rcParams.update({'font.family':'serif','font.size':11,
    'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':0.8,
    'xtick.direction':'in','ytick.direction':'in','figure.dpi':150})
try:
    from cuml.decomposition import PCA as cuPCA
    import cupy as cp
    CUDA_ML = True
    print("cuML available — GPU PCA enabled")
except ImportError:
    CUDA_ML = False
    print("cuML not available — CPU PCA fallback")
N_SAMPLES        = 25_000
N_BIAS_SUB       = 5_000
N_SHUFFLES       = 500
STIM_ONSET       = 100
WINDOW_SIZE      = 15
STEP_SIZE        = 5
TIME_BINS        = np.arange(0, 300-WINDOW_SIZE, STEP_SIZE)
EMERGE_GATE_NSEM = 3.0
CLUSTER_ALPHA    = 0.05
SWEEP_CONFIGS    = [(10,2),(15,4),(20,5),(30,5),(50,10)]
ROIS             = ['V1','IT','V4']
ROI_COLORS       = {'V1':'#1f77b4','IT':'#2ca02c','V4':'#ff7f0e'}
LAYER_NAMES_ORDERED = ['layer1_pool', 'layer2_pool', 'layer3_pool', 'layer4_pool', 'avgpool']
LAYER_LABELS = {
    'layer1_pool': 'ResNet18-Layer1 (64ch, spatially pooled)',
    'layer2_pool': 'ResNet18-Layer2 (128ch, spatially pooled)',
    'layer3_pool': 'ResNet18-Layer3 (256ch, spatially pooled)',
    'layer4_pool': 'ResNet18-Layer4 (512ch, spatially pooled)',
    'avgpool':     'ResNet18-AvgPool (512-d, original)',
}
MAIN_LAYER = 'avgpool'
CACHE_PATH  = '/content/monkey_full_cleaned_normalized.h5'
ACTUAL_ROOT = '/content/drive/MyDrive/THINGS_images_unzipped/object_images'
DRIVE_OUT   = '/content/drive/MyDrive/neurips_v9_results'
os.makedirs(DRIVE_OUT, exist_ok=True)
print("="*60)
print("Loading neural data...")
t0 = time.time()
with h5py.File(CACHE_PATH,'r') as hf:
    clean_data = hf['data'][:].astype(np.float64)
T_total, N_trials, N_units = clean_data.shape
print(f"  Loaded: {clean_data.shape}  ({time.time()-t0:.1f}s)")
assert N_units==1024 and T_total==300 and N_trials>=25000
roi_clean = {'V1': clean_data[:,:,0:512],
             'IT': clean_data[:,:,512:832],
             'V4': clean_data[:,:,832:]}
assert roi_clean['V1'].shape[2]==512
assert roi_clean['IT'].shape[2]==320
assert roi_clean['V4'].shape[2]==192
print(f"  V1:{roi_clean['V1'].shape[2]}  IT:{roi_clean['IT'].shape[2]}  V4:{roi_clean['V4'].shape[2]}")
print("Resolving image paths...")
master_metadata['full_path'] = master_metadata['relative_path'].apply(
    lambda x: os.path.join(ACTUAL_ROOT, x))
valid_metadata = (master_metadata[master_metadata['full_path'].apply(os.path.exists)]
                  .copy().reset_index(drop=True))
print(f"  Valid images: {len(valid_metadata)}")
assert len(valid_metadata) >= N_SAMPLES,    f"Only {len(valid_metadata)} valid images but N_SAMPLES={N_SAMPLES}"
trials_all = valid_metadata['trial_idx'].values
print(f"  Using ALL {len(trials_all)} trials (no subsampling)")
print("\nExtracting CNN features (all ResNet-18 intermediate + final layers)...")
class THINGSDs(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            with Image.open(self.paths[i]) as img:
                return self.tf(img.convert('RGB'))
        except:
            return torch.zeros(3, 224, 224)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Device: {device}")
mdl = models.resnet18(weights='DEFAULT').to(device).eval()
layer_pool_buffers = {k: [] for k in LAYER_NAMES_ORDERED}
def _make_hook(name, spatial_collapse=True):
    def _hook(m, i, o):
        x = o
        if spatial_collapse and x.dim() == 4:
            x = x.mean(dim=[2, 3])
        else:
            x = x.view(x.size(0), -1)
        layer_pool_buffers[name].append(x.detach().cpu().numpy())
    return _hook
hooks = [
    mdl.layer1.register_forward_hook(_make_hook('layer1_pool', spatial_collapse=True)),
    mdl.layer2.register_forward_hook(_make_hook('layer2_pool', spatial_collapse=True)),
    mdl.layer3.register_forward_hook(_make_hook('layer3_pool', spatial_collapse=True)),
    mdl.layer4.register_forward_hook(_make_hook('layer4_pool', spatial_collapse=True)),
    mdl.avgpool.register_forward_hook(_make_hook('avgpool',     spatial_collapse=False)),
]
tf_cpu = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
normalize_gpu = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
dl = DataLoader(
    THINGSDs(valid_metadata['full_path'].tolist(), tf_cpu),
    batch_size=256,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=4,
    persistent_workers=True,
)
t0 = time.time()
with torch.no_grad():
    for b_idx, b in enumerate(dl):
        b = normalize_gpu(b.to(device, non_blocking=True))
        mdl(b)
        if (b_idx+1) % 5 == 0:
            print(f"  batch {b_idx+1}/{len(dl)}  ({time.time()-t0:.1f}s elapsed)")
for h in hooks:
    h.remove()
cnn_layer_feats = {}
for lname in LAYER_NAMES_ORDERED:
    raw = np.vstack(layer_pool_buffers[lname]).astype(np.float64)
    n_comp = min(50, raw.shape[1] - 1)
    cnn_layer_feats[lname] = PCA(n_components=n_comp).fit_transform(raw)
    print(f"  {lname}: raw {raw.shape} → PCA-{n_comp} {cnn_layer_feats[lname].shape}")
print(f"  Feature extraction complete  ({time.time()-t0:.1f}s)")
print("="*60)
def mp_pca(X):
    X   = np.asarray(X, dtype=np.float32)
    n, p = X.shape
    Xc  = (X - X.mean(0)).astype(np.float64)
    if CUDA_ML and p > 50:
        try:
            n_comp = min(p-1, n-1, 512)
            pca_gpu = cuPCA(n_components=n_comp, random_state=0)
            pca_gpu.fit(cp.asarray(Xc.astype(np.float32)))
            lam  = cp.asnumpy(pca_gpu.explained_variance_).astype(np.float64)
            vecs = cp.asnumpy(pca_gpu.components_).T.astype(np.float64)
        except Exception:
            cov = np.cov(Xc, rowvar=False)
            la, ve = eigh(cov); lam = la[::-1]; vecs = ve[:,::-1]
    else:
        cov = np.cov(Xc, rowvar=False)
        la, ve = eigh(cov); lam = la[::-1]; vecs = ve[:,::-1]
    gamma    = p / n
    sigma2   = float(np.median(lam[lam > 0]))
    lam_plus = sigma2 * (1.0 + np.sqrt(gamma))**2
    k        = max(1, int(np.sum(lam > lam_plus)))
    return Xc @ vecs[:, :k], k, lam
def mp_pca_from_pcs(pc_list):
    X = np.hstack(pc_list).astype(np.float64)
    n, p = X.shape
    Xc = X - X.mean(0)
    cov = np.cov(Xc, rowvar=False)
    la, ve = eigh(cov); lam = la[::-1]; vecs = ve[:,::-1]
    gamma    = p / n
    sigma2   = float(np.median(lam[lam > 0]))
    lam_plus = sigma2 * (1.0 + np.sqrt(gamma))**2
    k        = max(1, int(np.sum(lam > lam_plus)))
    return Xc @ vecs[:, :k], k, lam
def _mi_gauss(X, Y):
    X  = np.asarray(X, dtype=np.float64); Xc = X - X.mean(0)
    Y  = np.asarray(Y, dtype=np.float64); Yc = Y - Y.mean(0)
    n  = len(Xc); dx = Xc.shape[1]
    XY = np.hstack([Xc, Yc])
    C  = (XY.T @ XY) / (n - 1)
    Cx = C[:dx, :dx]; Cy = C[dx:, dx:]
    sx, ldx = slogdet(Cx)
    sy, ldy = slogdet(Cy)
    sc, ldc = slogdet(C)
    if sx<=0 or sy<=0 or sc<=0: return 0.0
    return max(0.0, 0.5*(ldx + ldy - ldc))
def _mi_gauss_batch_shuffles(X, Y, n_sh=N_SHUFFLES, rng=None):
    if rng is None: rng = np.random.default_rng()
    X  = np.asarray(X, dtype=np.float64); Xc = X - X.mean(0)
    Y  = np.asarray(Y, dtype=np.float64)
    n  = len(Xc); dx = Xc.shape[1]
    Cxx = (Xc.T @ Xc) / (n-1)
    sx, ldx = slogdet(Cxx)
    if sx <= 0: return np.zeros(n_sh)
    mis = np.zeros(n_sh)
    for sh in range(n_sh):
        Ys = Y[rng.permutation(n)]; Yc = Ys - Ys.mean(0)
        XY = np.hstack([Xc, Yc]); C = (XY.T @ XY) / (n-1)
        Cy = C[dx:, dx:]
        sy, ldy = slogdet(Cy); sc, ldc = slogdet(C)
        if sy<=0 or sc<=0: mis[sh]=0.0
        else: mis[sh] = max(0.0, 0.5*(ldx+ldy-ldc))
    return mis
def get_bias(S_list, T, n_sh=N_SHUFFLES, n_sub=N_BIAS_SUB, rng=None):
    if rng is None: rng = np.random.default_rng()
    n   = len(T)
    idx = rng.choice(n, min(n_sub,n), replace=False)
    Ss  = [S[idx] for S in S_list]; Ts = T[idx]
    bm  = np.zeros(len(S_list)); nm = np.zeros((len(S_list),n_sh))
    bt  = 0.0;                    nt = np.zeros(n_sh)
    for sh in range(n_sh):
        perm = rng.permutation(len(Ts)); Tp = Ts[perm]
        mv   = np.array([_mi_gauss(S, Tp) for S in Ss])
        mj   = _mi_gauss(np.hstack(Ss), Tp)
        bm  += mv; bt += mj; nm[:,sh] = mv; nt[sh] = mj
    return bm/n_sh, bt/n_sh, nm, nt
def get_bias_single(Z, T, n_sh=N_SHUFFLES, n_sub=N_BIAS_SUB, rng=None):
    if rng is None: rng = np.random.default_rng()
    n   = len(T)
    idx = rng.choice(n, min(n_sub,n), replace=False)
    Zs  = Z[idx]; Ts = T[idx]
    nt  = _mi_gauss_batch_shuffles(Zs, Ts, n_sh=n_sh, rng=rng)
    return float(np.mean(nt)), nt
def hybrid_pid(S_perpc, Z_joint, T, bias_mis, bias_total_perpc, bias_total_joint):
    mis = np.array([_mi_gauss(S,T) for S in S_perpc]) - bias_mis
    mis = np.maximum(0.0, mis)
    mi_joint    = max(0.0, _mi_gauss(Z_joint,T) - bias_total_joint)
    red         = float(np.min(mis))
    uniques     = [max(0.0, float(m)-red) for m in mis]
    syn         = max(0.0, mi_joint - red - sum(uniques))
    total_perpc = max(0.0, _mi_gauss(np.hstack(S_perpc),T) - bias_total_perpc)
    return {'Total':total_perpc,'Redundancy':red,'Synergy':syn,
            'Uniques':uniques,'MI_joint':mi_joint}
def block_pid(Z_pair_joint, Z_lo_pc, T, bias_mis2, bias_total_2src, bias_total_block_joint):
    Z_bj  = np.hstack([Z_pair_joint, Z_lo_pc])
    mis   = np.array([_mi_gauss(S,T) for S in [Z_pair_joint, Z_lo_pc]]) - bias_mis2
    mis   = np.maximum(0.0, mis)
    mi_bj = max(0.0, _mi_gauss(Z_bj,T) - bias_total_block_joint)
    mi_2s = max(0.0, _mi_gauss(Z_bj,T) - bias_total_2src)
    red   = float(np.min(mis))
    u_pair= max(0.0, float(mis[0])-red)
    u_lo  = max(0.0, float(mis[1])-red)
    syn   = max(0.0, mi_bj - red - u_pair - u_lo)
    return {'Total':mi_2s,'Redundancy':red,'Synergy':syn,
            'Unique_pair':u_pair,'Unique_lo':u_lo,'MI_block_joint':mi_bj}
def pairwise_pid(S_A, S_B, Z_pair_joint, T, bias_mis2, bias_total_2src, bias_total_pair_joint):
    mis   = np.array([_mi_gauss(S,T) for S in [S_A,S_B]]) - bias_mis2
    mis   = np.maximum(0.0, mis)
    mi_pj = max(0.0, _mi_gauss(Z_pair_joint,T) - bias_total_pair_joint)
    mi_2s = max(0.0, _mi_gauss(np.hstack([S_A,S_B]),T) - bias_total_2src)
    red   = float(np.min(mis))
    uA    = max(0.0, float(mis[0])-red)
    uB    = max(0.0, float(mis[1])-red)
    syn   = max(0.0, mi_pj - red - uA - uB)
    return {'Total':mi_2s,'Redundancy':red,'Synergy':syn,'Unique_A':uA,'Unique_B':uB}
def compute_manifold_geometry(X):
    X   = np.asarray(X, dtype=np.float64); Xc = X - X.mean(0)
    cov = np.cov(Xc, rowvar=False).astype(np.float64)
    lam = eigh(cov, eigvals_only=True).astype(np.float64)
    lam = np.sort(lam)[::-1]; lam = lam[lam>1e-10]
    if len(lam)==0: return 0.0,1.0,0.0
    pr   = float((np.sum(lam)**2)/np.sum(lam**2))
    pc1f = float(lam[0]/np.sum(lam))
    sv   = np.linalg.svd(Xc,compute_uv=False).astype(np.float64); sv=sv[sv>1e-10]
    cond = float(sv[0]/sv[-1]) if len(sv)>1 else 1.0
    return pr, cond, pc1f
def get_rsa_permtest(Xn, Xc, n_sub=500, n_rep=5, n_perm=200, seed=None):
    rng = np.random.default_rng(seed)
    n   = min(len(Xn), len(Xc)); rs=[]
    for _ in range(n_rep):
        idx = rng.choice(n, n_sub, replace=False)
        A = (Xn[idx]-Xn[idx].mean(0))/(Xn[idx].std(0)+1e-8)
        B = (Xc[idx]-Xc[idx].mean(0))/(Xc[idx].std(0)+1e-8)
        rdA=1-np.corrcoef(A); rdB=1-np.corrcoef(B)
        ti,tj=np.triu_indices(n_sub,k=1)
        r,_=spearmanr(rdA[ti,tj],rdB[ti,tj]); rs.append(float(r))
    obs_r = float(np.mean(rs))
    null_rs=[]
    idx_perm=rng.choice(n,n_sub,replace=False)
    A_p=(Xn[idx_perm]-Xn[idx_perm].mean(0))/(Xn[idx_perm].std(0)+1e-8)
    rdA_p=1-np.corrcoef(A_p)
    for _ in range(n_perm):
        shuf=rng.permutation(n_sub)
        B_s=(Xc[idx_perm[shuf]]-Xc[idx_perm[shuf]].mean(0))/(Xc[idx_perm[shuf]].std(0)+1e-8)
        rdB_s=1-np.corrcoef(B_s)
        r_null,_=spearmanr(rdA_p[ti,tj],rdB_s[ti,tj]); null_rs.append(float(r_null))
    pval=(np.sum(np.array(null_rs)>=obs_r)+1)/(n_perm+1)
    return obs_r, pval
def gaussianity_check(X, label, n_test=5000):
    rng=np.random.default_rng(0); idx=rng.choice(len(X),min(n_test,len(X)),replace=False)
    pcs=PCA(n_components=5).fit_transform(X[idx]); rows=[]
    for k in range(5):
        stat,p=normaltest(pcs[:,k])
        rows.append({'ROI':label,'PC':k+1,
                     'statistic':round(float(stat),4),'p_value':round(float(p),6)})
    return pd.DataFrame(rows)
def baseline_subtract_null_matrix(null_mat_raw, pre_mask):
    """Subtract each shuffle column's own pre-stim mean, mirroring _bs()."""
    pre_mean = null_mat_raw[pre_mask, :].mean(axis=0)
    return null_mat_raw - pre_mean[np.newaxis, :]
def omnibus_permtest(obs_arr, null_mat, post_mask):
    obs_mean   = float(np.mean(obs_arr[post_mask]))
    null_means = np.mean(null_mat[post_mask, :], axis=0)
    pval = (np.sum(null_means >= obs_mean) + 1) / (null_mat.shape[1] + 1)
    return obs_mean, null_means, float(pval)
def null_cluster_masses(null_mat, null_mean_per_bin, null_std_per_bin):
    threshold = null_mean_per_bin + 2.0 * null_std_per_bin
    masses = []
    for rep in range(null_mat.shape[1]):
        s   = null_mat[:, rep]
        sig = s > threshold
        mx  = 0.0; inc = False; cur = 0.0
        for v, f in zip(s, sig):
            if f:
                cur += abs(float(v)); inc = True
            else:
                if inc: mx = max(mx, cur)
                cur = 0.0; inc = False
        if inc: mx = max(mx, cur)
        masses.append(mx)
    return masses
def cluster_mass_test(obs_arr, null_mat, null_mean_per_bin, null_std_per_bin,
                      null_cluster_mass_dist, alpha=CLUSTER_ALPHA):
    threshold = null_mean_per_bin + 2.0 * null_std_per_bin
    clusters = []
    in_cl = False; start = 0
    for i, v in enumerate(obs_arr):
        if v > threshold[i] and not in_cl:
            in_cl = True; start = i
        elif (v <= threshold[i] or i == len(obs_arr)-1) and in_cl:
            end = i-1 if v <= threshold[i] else i
            clusters.append((start, end))
            in_cl = False
    if in_cl:
        clusters.append((start, len(obs_arr)-1))
    null_arr = np.array(null_cluster_mass_dist)
    onsets = []; peaks = []; pvals = []
    sig_mask = np.zeros(len(obs_arr), dtype=bool)
    for s, e in clusters:
        mass = float(np.sum(obs_arr[s:e+1]))
        pval = (np.sum(null_arr >= mass) + 1) / (len(null_arr) + 1)
        onsets.append(s); peaks.append(s + int(np.argmax(obs_arr[s:e+1]))); pvals.append(pval)
        if pval < alpha: sig_mask[s:e+1] = True
    return onsets, peaks, pvals, sig_mask, null_mean_per_bin, null_std_per_bin
def compute_null_stats_per_bin(null_mat):
    return null_mat.mean(axis=1), null_mat.std(axis=1)
def run_significance(null_mat_raw, obs_arr, pre_mask, post_mask, label=''):
    """Full corrected significance pipeline. Returns dict of all outputs."""
    null_mat = baseline_subtract_null_matrix(null_mat_raw, pre_mask)
    null_mean_pb, null_std_pb = compute_null_stats_per_bin(null_mat)
    omni_mean, omni_null, omni_pval = omnibus_permtest(obs_arr, null_mat, post_mask)
    ncm  = null_cluster_masses(null_mat, null_mean_pb, null_std_pb)
    (clo, clp, clpv, sig_mask, _, _) = cluster_mass_test(
        obs_arr, null_mat, null_mean_pb, null_std_pb, ncm)
    n_sig = sum(p < CLUSTER_ALPHA for p in clpv)
    print(f"    [{label}] omnibus p={omni_pval:.4f}  "
          f"clusters={len(clo)}  sig={n_sig}  "
          f"sig_bins={sig_mask.sum()}/{len(sig_mask)}")
    return dict(null_mat=null_mat, null_mean_pb=null_mean_pb, null_std_pb=null_std_pb,
                omni_mean=omni_mean, omni_null=omni_null, omni_pval=omni_pval,
                ncm=ncm, clo=clo, clp=clp, clpv=clpv, sig_mask=sig_mask)
def add_stim(ax, label=True):
    ax.axvline(STIM_ONSET, color='black', ls='--', lw=1.8, alpha=0.7, zorder=5,
               label='Stimulus onset' if label else None)
def shade_sig(ax, ta, mask, color='gold', alpha=0.25):
    for i, s in enumerate(mask):
        if s: ax.axvspan(ta[i]-STEP_SIZE/2, ta[i]+STEP_SIZE/2,
                         color=color, alpha=alpha, zorder=1)
def shade_null_band(ax, t_axis, null_mean, null_std, n_sd=2, color='grey', alpha=0.18):
    upper = null_mean + n_sd * null_std
    lower = null_mean - n_sd * null_std
    ax.fill_between(t_axis, lower, upper, color=color, alpha=alpha,
                    label=f'Shuffle null ±{n_sd}SD', zorder=0)
    ax.plot(t_axis, null_mean, color=color, lw=0.8, ls=':', alpha=0.6)
def save_csv(df, fname, title=None):
    path = os.path.join(DRIVE_OUT, fname)
    df.to_csv(path, index=False)
    if title: print(f"  Saved: {fname}  ({len(df)} rows)")
def save_fig(fig, fname):
    path = os.path.join(DRIVE_OUT, fname)
    fig.savefig(path, bbox_inches='tight', dpi=150)
    print(f"  Saved: {fname}")
n_tbins = len(TIME_BINS)
all_geometry_rows     = []
all_mpk_rows          = []
all_rsa_rows          = []
all_pid3_rows         = []
all_cluster_rows      = []
all_block_pid_rows    = []
all_pairwise_pid_rows = []
all_gauss_rows        = []
all_summary_rows      = []
plot_data = {}
t_grand = time.time()
for lname in LAYER_NAMES_ORDERED:
    cnn_feats_layer = cnn_layer_feats[lname]
    lbl             = LAYER_LABELS[lname]
    print("\n" + "="*60)
    print(f"LAYER: {lname}  |  {lbl}")
    print(f"  CNN feature shape: {cnn_feats_layer.shape}")
    print("="*60)
    rng = np.random.default_rng(0)
    geom          = {r:{'PR':[],'CN':[],'PC1_Frac':[]} for r in ROIS}
    rsa_res       = {r:{'r':[],'pval':[]} for r in ROIS}
    neu2img       = []
    null_total_list = [[] for _ in range(n_tbins)]
    null_syn_list   = [[] for _ in range(n_tbins)]
    block_pid_res = {'V1IT_vs_V4':[],'V1V4_vs_IT':[],'ITV4_vs_V1':[]}
    pair_pid_res  = {'V1_IT':[],'V1_V4':[],'IT_V4':[]}
    mp_k          = {r:[] for r in ROIS}
    mp_k_joint    = []
    mp_k_block    = {'V1IT_vs_V4':[],'V1V4_vs_IT':[],'ITV4_vs_V1':[]}
    mp_k_pair     = {'V1_IT':[],'V1_V4':[],'IT_V4':[]}
    gauss_rows_layer = []
    t_loop = time.time()
    print(f"  Main loop: {n_tbins} bins × {N_SHUFFLES} shuffles")
    for t_idx, t in enumerate(TIME_BINS):
        raw = {roi: np.mean(roi_clean[roi][t:t+WINDOW_SIZE, trials_all, :],
                            axis=0).astype(np.float64)
               for roi in ROIS}
        for roi in ROIS:
            pr, cond, pc1f = compute_manifold_geometry(raw[roi])
            geom[roi]['PR'].append(pr); geom[roi]['CN'].append(cond)
            geom[roi]['PC1_Frac'].append(pc1f)
        pc_roi = {}
        for roi in ROIS:
            pc_roi[roi], k_r, _ = mp_pca(raw[roi])
            mp_k[roi].append(k_r)
        if t_idx == 0:
            for roi in ROIS:
                df_g = gaussianity_check(pc_roi[roi], roi)
                df_g['Time_ms']     = float(t + WINDOW_SIZE/2)
                df_g['Layer']       = lname
                df_g['Layer_label'] = lbl
                gauss_rows_layer.append(df_g)
        for roi in ROIS:
            r_val, pval = get_rsa_permtest(
                pc_roi[roi], cnn_feats_layer,
                n_sub=500, n_rep=5, n_perm=200, seed=t_idx*100)
            rsa_res[roi]['r'].append(r_val)
            rsa_res[roi]['pval'].append(pval)
        S_perpc  = [pc_roi[r] for r in ROIS]
        Z_joint3, k_j3, _ = mp_pca_from_pcs(S_perpc)
        mp_k_joint.append(k_j3)
        bm3, bt3_perpc, nm3, nt3 = get_bias(S_perpc, cnn_feats_layer, rng=rng)
        bt3_joint, _             = get_bias_single(Z_joint3, cnn_feats_layer, rng=rng)
        pid3 = hybrid_pid(S_perpc, Z_joint3, cnn_feats_layer, bm3, bt3_perpc, bt3_joint)
        neu2img.append(pid3)
        null_total_list[t_idx].extend(nt3.tolist())
        null_syn_vals = []
        for sh in range(N_SHUFFLES):
            nm = nm3[:,sh]; nr = float(np.min(nm))
            ns = max(0.0, nt3[sh] - np.sum([max(0., float(m)-nr) for m in nm]) - nr)
            null_syn_vals.append(ns)
        null_syn_list[t_idx].extend(null_syn_vals)
        block_configs = {
            'V1IT_vs_V4': (['V1','IT'], 'V4'),
            'V1V4_vs_IT': (['V1','V4'], 'IT'),
            'ITV4_vs_V1': (['IT','V4'], 'V1'),
        }
        roi_idx = {'V1':0, 'IT':1, 'V4':2}
        for bk, (pair_rois, lo_roi) in block_configs.items():
            Z_pair_joint, k_pair, _ = mp_pca_from_pcs([pc_roi[r] for r in pair_rois])
            Z_lo_pc = pc_roi[lo_roi]
            mp_k_block[bk].append(k_pair)
            bm2, bt2, _, _ = get_bias([Z_pair_joint, Z_lo_pc], cnn_feats_layer, rng=rng)
            bt_bj, _       = get_bias_single(
                np.hstack([Z_pair_joint, Z_lo_pc]), cnn_feats_layer, rng=rng)
            block_pid_res[bk].append(
                block_pid(Z_pair_joint, Z_lo_pc, cnn_feats_layer, bm2, bt2, bt_bj))
        pair_configs = {'V1_IT':('V1','IT'), 'V1_V4':('V1','V4'), 'IT_V4':('IT','V4')}
        for pk, (rA, rB) in pair_configs.items():
            Z_pair2, k_p2, _ = mp_pca_from_pcs([pc_roi[rA], pc_roi[rB]])
            mp_k_pair[pk].append(k_p2)
            bias_marginals = np.array([bm3[roi_idx[rA]], bm3[roi_idx[rB]]])
            bt_2src = float(np.mean([bm3[roi_idx[rA]], bm3[roi_idx[rB]]]))
            bt_pj, _ = get_bias_single(Z_pair2, cnn_feats_layer, rng=rng)
            pair_pid_res[pk].append(
                pairwise_pid(pc_roi[rA], pc_roi[rB], Z_pair2,
                             cnn_feats_layer, bias_marginals, bt_2src, bt_pj))
        elapsed = time.time()-t_loop
        eta     = (elapsed/(t_idx+1))*(n_tbins-t_idx-1)
        if (t_idx+1) % 5 == 0:
            print(f"    bin {t_idx+1}/{n_tbins}  t={t}ms  "
                  f"elapsed={elapsed:.0f}s  ETA={eta:.0f}s")
    print(f"  Loop done in {(time.time()-t_loop)/60:.1f} min")
    t_axis = TIME_BINS + WINDOW_SIZE/2.0
    pre    = t_axis < STIM_ONSET
    post   = t_axis >= STIM_ONSET
    t_post = t_axis[post]
    def _bs(arr):
        b = float(np.mean(arr[pre])) if np.any(pre) else 0.0
        return arr - b
    geom_mu = {r:{k:np.array(v) for k,v in geom[r].items()} for r in ROIS}
    rsa_r   = {r: _bs(np.array(rsa_res[r]['r']))  for r in ROIS}
    rsa_pv  = {r: np.array(rsa_res[r]['pval'])    for r in ROIS}
    rsa_sig = {r: rsa_pv[r] < 0.05               for r in ROIS}
    tot_arr = _bs(np.array([p['Total']      for p in neu2img]))
    red_arr = _bs(np.array([p['Redundancy'] for p in neu2img]))
    syn_arr = _bs(np.array([p['Synergy']    for p in neu2img]))
    uv1_arr = _bs(np.array([p['Uniques'][0] for p in neu2img]))
    uit_arr = _bs(np.array([p['Uniques'][1] for p in neu2img]))
    uv4_arr = _bs(np.array([p['Uniques'][2] for p in neu2img]))
    mj_arr  = _bs(np.array([p['MI_joint']   for p in neu2img]))
    def blk(bk, c): return _bs(np.array([p[c] for p in block_pid_res[bk]]))
    blk_tot   = {bk: blk(bk,'Total')       for bk in block_pid_res}
    blk_red   = {bk: blk(bk,'Redundancy')  for bk in block_pid_res}
    blk_syn   = {bk: blk(bk,'Synergy')     for bk in block_pid_res}
    blk_upair = {bk: blk(bk,'Unique_pair') for bk in block_pid_res}
    blk_ulo   = {bk: blk(bk,'Unique_lo')   for bk in block_pid_res}
    def pw(pk, c): return _bs(np.array([p[c] for p in pair_pid_res[pk]]))
    pair_tot = {pk: pw(pk,'Total')      for pk in pair_pid_res}
    pair_red = {pk: pw(pk,'Redundancy') for pk in pair_pid_res}
    pair_syn = {pk: pw(pk,'Synergy')    for pk in pair_pid_res}
    pair_uA  = {pk: pw(pk,'Unique_A')   for pk in pair_pid_res}
    pair_uB  = {pk: pw(pk,'Unique_B')   for pk in pair_pid_res}
    def gated_emerge(syn, tot):
        pre_std = float(np.std(tot[pre])) if np.any(pre) else 1e-6
        gate    = EMERGE_GATE_NSEM * pre_std
        valid   = tot > gate
        return np.where(valid, syn/np.maximum(tot,1e-12), np.nan)
    em_arr = gated_emerge(syn_arr, tot_arr)
    print(f"  Significance tests for {lname}...")
    null_mat_tot_raw = np.array([null_total_list[ti] for ti in range(n_tbins)])
    null_mat_syn_raw = np.array([null_syn_list[ti]   for ti in range(n_tbins)])
    sig_tot = run_significance(null_mat_tot_raw, tot_arr, pre, post,
                               label=f'{lname}-TotalMI')
    sig_syn = run_significance(null_mat_syn_raw, syn_arr, pre, post,
                               label=f'{lname}-Synergy')
    null_mean_tot    = sig_tot['null_mean_pb']
    null_std_tot     = sig_tot['null_std_pb']
    sig_cluster_tot  = sig_tot['sig_mask']
    omni_mean_tot    = sig_tot['omni_mean']
    omni_pval_tot    = sig_tot['omni_pval']
    clo_tot          = sig_tot['clo']
    clp_tot          = sig_tot['clp']
    clpv_tot         = sig_tot['clpv']
    null_mean_syn    = sig_syn['null_mean_pb']
    null_std_syn     = sig_syn['null_std_pb']
    sig_cluster_syn  = sig_syn['sig_mask']
    omni_mean_syn    = sig_syn['omni_mean']
    omni_pval_syn    = sig_syn['omni_pval']
    clo_syn          = sig_syn['clo']
    clp_syn          = sig_syn['clp']
    clpv_syn         = sig_syn['clpv']
    if lname == MAIN_LAYER:
        plot_data = dict(
            t_axis=t_axis, pre=pre, post=post, t_post=t_post,
            tot_arr=tot_arr, red_arr=red_arr, syn_arr=syn_arr,
            uv1_arr=uv1_arr, uit_arr=uit_arr, uv4_arr=uv4_arr,
            mj_arr=mj_arr, em_arr=em_arr,
            null_mean_tot=null_mean_tot, null_std_tot=null_std_tot,
            null_mean_syn=null_mean_syn, null_std_syn=null_std_syn,
            sig_cluster_tot=sig_cluster_tot, sig_cluster_syn=sig_cluster_syn,
            omni_mean_tot=omni_mean_tot, omni_pval_tot=omni_pval_tot,
            omni_mean_syn=omni_mean_syn, omni_pval_syn=omni_pval_syn,
            clo_tot=clo_tot, clp_tot=clp_tot, clpv_tot=clpv_tot,
            clo_syn=clo_syn, clp_syn=clp_syn, clpv_syn=clpv_syn,
            blk_tot=blk_tot, blk_red=blk_red, blk_syn=blk_syn,
            blk_upair=blk_upair, blk_ulo=blk_ulo,
            pair_tot=pair_tot, pair_red=pair_red, pair_syn=pair_syn,
            pair_uA=pair_uA, pair_uB=pair_uB,
            geom_mu=geom_mu, rsa_r=rsa_r, rsa_pv=rsa_pv, rsa_sig=rsa_sig,
        )
    def _lp():
        return {'Layer': lname, 'Layer_label': lbl}
    for roi in ROIS:
        for ti, tv in enumerate(t_axis):
            all_geometry_rows.append({**_lp(), 'ROI': roi,
                'Time_ms':   round(float(tv), 1),
                'PR':        round(float(geom_mu[roi]['PR'][ti]),     6),
                'CN':        round(float(geom_mu[roi]['CN'][ti]),     6),
                'PC1_Frac':  round(float(geom_mu[roi]['PC1_Frac'][ti]), 6)})
    for roi in ROIS:
        for ti, tv in enumerate(t_axis):
            all_mpk_rows.append({**_lp(), 'Source': roi,
                'Time_ms': round(float(tv),1), 'k': int(mp_k[roi][ti])})
    for ti, tv in enumerate(t_axis):
        all_mpk_rows.append({**_lp(), 'Source': 'Joint_all3',
            'Time_ms': round(float(tv),1), 'k': int(mp_k_joint[ti])})
    for bk in block_pid_res:
        for ti, tv in enumerate(t_axis):
            all_mpk_rows.append({**_lp(), 'Source': f'Block_{bk}',
                'Time_ms': round(float(tv),1), 'k': int(mp_k_block[bk][ti])})
    for pk in pair_pid_res:
        for ti, tv in enumerate(t_axis):
            all_mpk_rows.append({**_lp(), 'Source': f'Pair_{pk}',
                'Time_ms': round(float(tv),1), 'k': int(mp_k_pair[pk][ti])})
    for roi in ROIS:
        for ti, tv in enumerate(t_axis):
            all_rsa_rows.append({**_lp(), 'ROI': roi,
                'Time_ms':    round(float(tv), 1),
                'Spearman_r': round(float(rsa_r[roi][ti]),  6),
                'pval_perm':  round(float(rsa_pv[roi][ti]), 6),
                'Sig_p05':    bool(rsa_sig[roi][ti])})
    for ti, tv in enumerate(t_axis):
        all_pid3_rows.append({**_lp(),
            'Time_ms':            round(float(tv), 1),
            'Total':              round(float(tot_arr[ti]), 6),
            'Redundancy':         round(float(red_arr[ti]), 6),
            'Synergy':            round(float(syn_arr[ti]), 6),
            'UniqueV1':           round(float(uv1_arr[ti]), 6),
            'UniqueIT':           round(float(uit_arr[ti]), 6),
            'UniqueV4':           round(float(uv4_arr[ti]), 6),
            'MI_joint':           round(float(mj_arr[ti]),  6),
            'EmergenceIndex':     float('nan') if np.isnan(em_arr[ti])
                                  else round(float(em_arr[ti]), 6),
            'null_mean_Total':    round(float(null_mean_tot[ti]), 6),
            'null_std_Total':     round(float(null_std_tot[ti]),  6),
            'null_mean_Syn':      round(float(null_mean_syn[ti]), 6),
            'null_std_Syn':       round(float(null_std_syn[ti]),  6),
            'sig_cluster_Total':  bool(sig_cluster_tot[ti]),
            'sig_cluster_Syn':    bool(sig_cluster_syn[ti]),
            'omnibus_pval_Total': round(float(omni_pval_tot), 6),
            'omnibus_pval_Syn':   round(float(omni_pval_syn), 6),
            'omnibus_mean_Total': round(float(omni_mean_tot), 6),
            'omnibus_mean_Syn':   round(float(omni_mean_syn), 6),
        })
    for k, (onset, peak, pval) in enumerate(zip(clo_tot, clp_tot, clpv_tot)):
        all_cluster_rows.append({**_lp(), 'Metric': 'TotalMI', 'Cluster': k,
            'Onset_ms':     round(float(t_axis[onset]), 1),
            'Peak_ms':      round(float(t_axis[peak]),  1),
            'Peak_val':     round(float(tot_arr[peak]), 6),
            'Cluster_mass': round(float(np.sum(tot_arr[onset:peak+1])), 6),
            'Cluster_pval': round(float(pval), 6),
            'Significant':  bool(pval < CLUSTER_ALPHA)})
    for k, (onset, peak, pval) in enumerate(zip(clo_syn, clp_syn, clpv_syn)):
        all_cluster_rows.append({**_lp(), 'Metric': 'Synergy', 'Cluster': k,
            'Onset_ms':     round(float(t_axis[onset]), 1),
            'Peak_ms':      round(float(t_axis[peak]),  1),
            'Peak_val':     round(float(syn_arr[peak]), 6),
            'Cluster_mass': round(float(np.sum(syn_arr[onset:peak+1])), 6),
            'Cluster_pval': round(float(pval), 6),
            'Significant':  bool(pval < CLUSTER_ALPHA)})
    all_cluster_rows.append({**_lp(), 'Metric': 'TotalMI_omnibus', 'Cluster': 0,
        'Onset_ms': float('nan'), 'Peak_ms': float('nan'),
        'Peak_val': round(omni_mean_tot, 6), 'Cluster_mass': float('nan'),
        'Cluster_pval': round(omni_pval_tot, 6),
        'Significant': bool(omni_pval_tot < CLUSTER_ALPHA)})
    all_cluster_rows.append({**_lp(), 'Metric': 'Synergy_omnibus', 'Cluster': 0,
        'Onset_ms': float('nan'), 'Peak_ms': float('nan'),
        'Peak_val': round(omni_mean_syn, 6), 'Cluster_mass': float('nan'),
        'Cluster_pval': round(omni_pval_syn, 6),
        'Significant': bool(omni_pval_syn < CLUSTER_ALPHA)})
    block_meta_save = [
        ('V1IT_vs_V4', 'V1+IT vs V4', 'V4',  'V1+IT'),
        ('V1V4_vs_IT', 'V1+V4 vs IT', 'IT',  'V1+V4'),
        ('ITV4_vs_V1', 'IT+V4 vs V1', 'V1',  'IT+V4'),
    ]
    for bk, title, lo_label, pair_label in block_meta_save:
        for ti, tv in enumerate(t_axis):
            all_block_pid_rows.append({**_lp(), 'Config': title,
                'Time_ms':                    round(float(tv), 1),
                'Total':                      round(float(blk_tot[bk][ti]),   6),
                'Redundancy':                 round(float(blk_red[bk][ti]),   6),
                f'Unique_{pair_label}':       round(float(blk_upair[bk][ti]), 6),
                f'Unique_{lo_label}':         round(float(blk_ulo[bk][ti]),   6),
                'Synergy':                    round(float(blk_syn[bk][ti]),   6)})
    pair_meta_save = [
        ('V1_IT', 'V1 vs IT', 'V1', 'IT'),
        ('V1_V4', 'V1 vs V4', 'V1', 'V4'),
        ('IT_V4', 'IT vs V4', 'IT', 'V4'),
    ]
    for pk, label, lA, lB in pair_meta_save:
        for ti, tv in enumerate(t_axis):
            all_pairwise_pid_rows.append({**_lp(), 'Pair': label,
                'Time_ms':        round(float(tv), 1),
                'Total':          round(float(pair_tot[pk][ti]), 6),
                'Redundancy':     round(float(pair_red[pk][ti]), 6),
                f'Unique_{lA}':   round(float(pair_uA[pk][ti]),  6),
                f'Unique_{lB}':   round(float(pair_uB[pk][ti]),  6),
                'Synergy':        round(float(pair_syn[pk][ti]), 6)})
    all_gauss_rows.extend(gauss_rows_layer)
    def _pk_s(arr, label_s):
        p = arr[post]; i = int(np.argmax(p))
        c = np.where(arr[post] > 0)[0]
        onset = round(float(t_post[c[0]]), 1) if len(c) > 0 else float('nan')
        return {**_lp(), 'Metric': label_s,
                'Peak_val': round(float(p[i]), 6),
                'Peak_ms':  round(float(t_post[i]), 1), 'Onset_ms': onset}
    def _tr_s(arr, label_s):
        p = arr[post]; i = int(np.argmin(p))
        return {**_lp(), 'Metric': label_s,
                'Trough_val': round(float(p[i]), 6),
                'Trough_ms':  round(float(t_post[i]), 1)}
    for roi in ROIS:
        all_summary_rows.append(_tr_s(geom_mu[roi]['PR'], f'PR_trough_{roi}'))
        all_summary_rows.append(_pk_s(geom_mu[roi]['PR'], f'PR_peak_{roi}'))
        all_summary_rows.append(_pk_s(geom_mu[roi]['CN'], f'CN_peak_{roi}'))
        all_summary_rows.append(_pk_s(rsa_r[roi],         f'RSA_peak_{roi}'))
        sig_bins  = np.where(rsa_sig[roi] & post)[0]
        rsa_onset = round(float(t_post[sig_bins[0]]), 1) if len(sig_bins) > 0 else float('nan')
        all_summary_rows.append({**_lp(), 'Metric': f'RSA_onset_{roi}',
            'Onset_ms': rsa_onset, 'Peak_val': float('nan'),
            'Peak_ms':  float('nan'), 'Trough_val': float('nan'), 'Trough_ms': float('nan')})
    all_summary_rows.append(_pk_s(tot_arr, 'TotalMI_peak'))
    all_summary_rows.append(_pk_s(syn_arr, 'Synergy_peak'))
    for metric, arr, clo_, clpv_ in [('TotalMI', tot_arr, clo_tot, clpv_tot),
                                      ('Synergy', syn_arr, clo_syn, clpv_syn)]:
        sig_cl = [(o, p) for o, p in zip(clo_, clpv_) if p < CLUSTER_ALPHA]
        onset  = round(float(t_axis[sig_cl[0][0]]), 1) if sig_cl else float('nan')
        n_sig  = int(sum(p < CLUSTER_ALPHA for p in clpv_))
        omni_p = omni_pval_tot if metric == 'TotalMI' else omni_pval_syn
        all_summary_rows.append({**_lp(), 'Metric': f'{metric}_cluster_onset',
            'Onset_ms': onset, 'N_sig_clusters': n_sig,
            'omnibus_pval': round(omni_p, 6),
            'Peak_val': float('nan'), 'Peak_ms': float('nan'),
            'Trough_val': float('nan'), 'Trough_ms': float('nan')})
    for k, (onset, peak, pval) in enumerate(zip(clo_tot, clp_tot, clpv_tot)):
        all_summary_rows.append({**_lp(), 'Metric': f'TotalMI_cluster{k}',
            'Onset_ms':    round(float(t_axis[onset]), 1),
            'Peak_ms':     round(float(t_axis[peak]),  1),
            'Peak_val':    round(float(tot_arr[peak]), 6),
            'Cluster_pval':round(float(pval), 6),
            'Significant': bool(pval < CLUSTER_ALPHA)})
    for k, (onset, peak, pval) in enumerate(zip(clo_syn, clp_syn, clpv_syn)):
        all_summary_rows.append({**_lp(), 'Metric': f'Synergy_cluster{k}',
            'Onset_ms':    round(float(t_axis[onset]), 1),
            'Peak_ms':     round(float(t_axis[peak]),  1),
            'Peak_val':    round(float(syn_arr[peak]), 6),
            'Cluster_pval':round(float(pval), 6),
            'Significant': bool(pval < CLUSTER_ALPHA)})
    all_summary_rows.append({**_lp(), 'Metric': 'Omnibus_TotalMI',
        'omnibus_obs_mean': round(omni_mean_tot, 6),
        'omnibus_pval':     round(omni_pval_tot, 6),
        'Significant':      bool(omni_pval_tot < CLUSTER_ALPHA)})
    all_summary_rows.append({**_lp(), 'Metric': 'Omnibus_Synergy',
        'omnibus_obs_mean': round(omni_mean_syn, 6),
        'omnibus_pval':     round(omni_pval_syn, 6),
        'Significant':      bool(omni_pval_syn < CLUSTER_ALPHA)})
    for bk, title, lo_label, pair_label in block_meta_save:
        all_summary_rows.append(_pk_s(blk_syn[bk],   f'BlockSyn_peak_{bk}'))
        all_summary_rows.append(_pk_s(blk_upair[bk], f'BlockUniquePair_peak_{bk}'))
        all_summary_rows.append(_pk_s(blk_ulo[bk],   f'BlockUniqueLO_peak_{bk}'))
    for pk, label, *_ in pair_meta_save:
        all_summary_rows.append(_pk_s(pair_syn[pk], f'PairSyn_peak_{pk}'))
        c = np.where((pair_syn[pk] > 0) & post)[0]
        onset = round(float(t_post[c[0]]), 1) if len(c) > 0 else float('nan')
        all_summary_rows.append({**_lp(), 'Metric': f'PairSyn_onset_{pk}',
            'Onset_ms': onset, 'Peak_val': float('nan'), 'Peak_ms': float('nan'),
            'Trough_val': float('nan'), 'Trough_ms': float('nan')})
    print(f"  Layer {lname} fully accumulated.")
print("\nSaving all stacked CSVs to Drive...")
save_csv(pd.DataFrame(all_geometry_rows),     'table_geometry.csv',           'Geometry')
save_csv(pd.DataFrame(all_mpk_rows),          'table_mp_dimensionality.csv',  'MP-k')
save_csv(pd.DataFrame(all_rsa_rows),          'table_rsa.csv',                'RSA')
save_csv(pd.DataFrame(all_pid3_rows),         'table_3source_pid.csv',        '3-source PID')
save_csv(pd.DataFrame(all_cluster_rows),      'table_cluster_stats.csv',      'Cluster stats')
save_csv(pd.DataFrame(all_block_pid_rows),    'table_block_pid.csv',          'Block PID')
save_csv(pd.DataFrame(all_pairwise_pid_rows), 'table_pairwise_pid.csv',       'Pairwise PID')
save_csv(pd.DataFrame(all_summary_rows),      'table_summary_statistics.csv', 'Summary stats')
if all_gauss_rows:
    save_csv(pd.concat(all_gauss_rows, ignore_index=True),
             'table_gaussianity.csv', 'Gaussianity')
print("\nRunning window-stride robustness sweep (avgpool, 3-way PID only)...")
rng_sw    = np.random.default_rng(42)
SWEEP_N   = 5000
idx_sw    = rng_sw.choice(len(trials_all), SWEEP_N, replace=False)
trials_sw = trials_all[idx_sw]
cnn_sw    = cnn_layer_feats[MAIN_LAYER][idx_sw]
sweep_rows = []
for ws, ss in SWEEP_CONFIGS:
    print(f"  W{ws}/S{ss}...")
    tbins_sw = np.arange(0,300-ws,ss); taxis_sw = tbins_sw + ws/2.0
    pre_sw   = taxis_sw < STIM_ONSET;  post_sw  = taxis_sw >= STIM_ONSET
    tot_sw=[]; syn_sw=[]; rsa_sw=[]
    for t in tbins_sw:
        raw_sw = {roi: np.mean(roi_clean[roi][t:t+ws, trials_sw, :],
                               axis=0).astype(np.float64) for roi in ROIS}
        pc_sw  = {roi: mp_pca(raw_sw[roi])[0] for roi in ROIS}
        Zj_sw, _, _ = mp_pca_from_pcs([pc_sw[r] for r in ROIS])
        S_sw = [pc_sw[r] for r in ROIS]
        bm_sw,bt_sw,_,_ = get_bias(S_sw, cnn_sw, n_sh=10, n_sub=1000, rng=rng_sw)
        bt_j_sw, _      = get_bias_single(Zj_sw, cnn_sw, n_sh=10, n_sub=1000, rng=rng_sw)
        pid_sw = hybrid_pid(S_sw, Zj_sw, cnn_sw, bm_sw, bt_sw, bt_j_sw)
        tot_sw.append(pid_sw['Total']); syn_sw.append(pid_sw['Synergy'])
        rsa_sw.append(get_rsa_permtest(pc_sw['IT'],cnn_sw,n_sub=300,n_rep=3,n_perm=50)[0])
    tot_sw=np.array(tot_sw); syn_sw=np.array(syn_sw); rsa_sw=np.array(rsa_sw)
    for arr in [tot_sw, syn_sw, rsa_sw]:
        b = float(np.mean(arr[pre_sw])) if np.any(pre_sw) else 0.0; arr -= b
    def _pk(a,m):   return float(np.max(a[m])) if np.any(m) else float('nan')
    def _lat(a,pr,po,tax):
        ps = float(np.std(a[pr])) if np.any(pr) else 1e-9
        c  = np.where((a>2*ps) & po)[0]
        return round(float(tax[c[0]]),1) if len(c)>0 else float('nan')
    def _pkl(a,po,tax):
        pi = np.where(po)[0]
        return round(float(tax[pi[np.argmax(a[pi])]]),1) if len(pi)>0 else float('nan')
    sweep_rows.append({'Window_ms':ws, 'Stride_ms':ss, 'N_bins':len(tbins_sw),
                       'Peak_TotalMI':     round(_pk(tot_sw, post_sw), 6),
                       'Peak_Synergy':     round(_pk(syn_sw, post_sw), 6),
                       'Peak_RSA_IT':      round(_pk(rsa_sw, post_sw), 6),
                       'Onset_TotalMI_ms': _lat(tot_sw,pre_sw,post_sw,taxis_sw),
                       'Onset_Synergy_ms': _lat(syn_sw,pre_sw,post_sw,taxis_sw),
                       'Peak_TotalMI_ms':  _pkl(tot_sw, post_sw, taxis_sw),
                       'Peak_Synergy_ms':  _pkl(syn_sw, post_sw, taxis_sw)})
    print(f"    peak_tot={sweep_rows[-1]['Peak_TotalMI']:.4f}  "
          f"peak_syn={sweep_rows[-1]['Peak_Synergy']:.4f}")
save_csv(pd.DataFrame(sweep_rows), 'table_window_stride_robustness.csv', 'Window-stride sweep')
print("\nGenerating plots (avgpool / main layer only)...")
t_axis        = plot_data['t_axis']
pre           = plot_data['pre'];             post          = plot_data['post']
t_post        = plot_data['t_post']
tot_arr       = plot_data['tot_arr'];         red_arr       = plot_data['red_arr']
syn_arr       = plot_data['syn_arr'];         uv1_arr       = plot_data['uv1_arr']
uit_arr       = plot_data['uit_arr'];         uv4_arr       = plot_data['uv4_arr']
mj_arr        = plot_data['mj_arr'];          em_arr        = plot_data['em_arr']
null_mean_tot = plot_data['null_mean_tot'];   null_std_tot  = plot_data['null_std_tot']
null_mean_syn = plot_data['null_mean_syn'];   null_std_syn  = plot_data['null_std_syn']
sig_cluster_tot = plot_data['sig_cluster_tot']
sig_cluster_syn = plot_data['sig_cluster_syn']
omni_mean_tot   = plot_data['omni_mean_tot']; omni_pval_tot = plot_data['omni_pval_tot']
omni_mean_syn   = plot_data['omni_mean_syn']; omni_pval_syn = plot_data['omni_pval_syn']
clo_tot  = plot_data['clo_tot'];  clp_tot  = plot_data['clp_tot'];  clpv_tot = plot_data['clpv_tot']
clo_syn  = plot_data['clo_syn'];  clp_syn  = plot_data['clp_syn'];  clpv_syn = plot_data['clpv_syn']
blk_tot  = plot_data['blk_tot'];  blk_red  = plot_data['blk_red'];  blk_syn  = plot_data['blk_syn']
blk_upair= plot_data['blk_upair']; blk_ulo = plot_data['blk_ulo']
geom_mu  = plot_data['geom_mu'];  rsa_r    = plot_data['rsa_r']
rsa_pv   = plot_data['rsa_pv'];   rsa_sig  = plot_data['rsa_sig']
fig3, axes3 = plt.subplots(1,3,figsize=(24,8)); fig3.patch.set_facecolor('#fafafa')
ax = axes3[0]
ax.stackplot(t_axis,
             np.maximum(red_arr,0), np.maximum(uv1_arr,0),
             np.maximum(uit_arr,0), np.maximum(uv4_arr,0), np.maximum(syn_arr,0),
             labels=['Redundancy','Unique V1 (per-ROI PC)',
                     'Unique IT (per-ROI PC)','Unique V4 (per-ROI PC)',
                     'Synergy (joint PC of per-ROI PCs)'],
             colors=['#7f7f7f',ROI_COLORS['V1'],ROI_COLORS['IT'],
                     ROI_COLORS['V4'],'#d62728'], alpha=0.85)
shade_sig(ax, t_axis, sig_cluster_tot, color='gold', alpha=0.18)
ax.set_title("3-source Hybrid PID: Neu→CNN\n"
             "Joint = 2nd-level MP-PCA on per-ROI PCs\n"
             "(electrode-count-balanced inter-area structure)",
             fontsize=10, fontweight='bold')
ax.set_ylabel("MI (nats, baseline-subtracted)"); ax.set_xlabel("Time (ms)")
ax.legend(loc='upper left', fontsize='x-small', frameon=False)
add_stim(ax, False); ax.set_facecolor('#fafafa'); ax.grid(True,ls=':',lw=0.5,alpha=0.5)
ax2 = axes3[1]
shade_null_band(ax2, t_axis, null_mean_tot, null_std_tot, n_sd=2, color='grey', alpha=0.20)
ax2.plot(t_axis, tot_arr, 'k-', lw=2.5, label='Total MI')
shade_sig(ax2, t_axis, sig_cluster_tot, color='gold', alpha=0.25)
for onset, peak, pval in zip(clo_tot, clp_tot, clpv_tot):
    if pval < CLUSTER_ALPHA:
        ax2.annotate(f'cluster p={pval:.3f}',
                     xy=(t_axis[peak], tot_arr[peak]),
                     xytext=(t_axis[peak], tot_arr[peak]*1.12),
                     arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5),
                     fontsize=9, color='darkred', ha='center')
ax2.plot(t_axis, syn_arr, color='#d62728', lw=2.0, ls='--', label='Synergy')
for onset, peak, pval in zip(clo_syn, clp_syn, clpv_syn):
    if pval < CLUSTER_ALPHA:
        ax2.annotate(f'cluster p={pval:.3f}',
                     xy=(t_axis[peak], syn_arr[peak]),
                     xytext=(t_axis[peak], syn_arr[peak]*1.25),
                     arrowprops=dict(arrowstyle='->', color='#8b0000', lw=1.5),
                     fontsize=9, color='#8b0000', ha='center')
ax2.set_title(
    f"Total MI + Synergy\n"
    f"Cluster-mass permutation (gold) | Omnibus: TotMI p={omni_pval_tot:.3f}, "
    f"Syn p={omni_pval_syn:.3f}",
    fontsize=9, fontweight='bold')
ax2.set_ylabel("MI (nats, baseline-sub)"); ax2.set_xlabel("Time (ms)")
add_stim(ax2); ax2.legend(frameon=False, fontsize=9)
ax2.set_facecolor('#fafafa'); ax2.grid(True,ls=':',lw=0.5,alpha=0.5)
ax3 = axes3[2]
valid_em = ~np.isnan(em_arr)
if np.any(valid_em):
    tv2 = t_axis[valid_em]; ev = em_arr[valid_em]
    ax3.plot(tv2, ev, 'r-', lw=2.2, label='Gated Emergence Index')
    ax3.fill_between(tv2, np.zeros_like(ev), ev, color='r', alpha=0.15)
ax3.set_title(f"Gated Emergence Index (syn/total)\n"
              f"[NaN where total<{EMERGE_GATE_NSEM}×pre-stim noise]",
              fontsize=10, fontweight='bold')
ax3.set_ylabel("Synergy / Total MI"); ax3.set_xlabel("Time (ms)")
add_stim(ax3, False); ax3.legend(frameon=False)
ax3.set_facecolor('#fafafa'); ax3.grid(True,ls=':',lw=0.5,alpha=0.5)
fig3.suptitle(
    "3-source Hybrid PID: per-ROI MP-PCA (unique subspaces) + "
    "2nd-level MP-PCA on concatenated PCs (joint/synergy)\n"
    f"ALL {len(trials_all)} trials | {N_SHUFFLES} shuffles | "
    f"Significance: cluster-mass permutation + omnibus permutation (CORRECTED null scale) | "
    f"bias subsampled N={N_BIAS_SUB}\n"
    "V1:509 elec | IT:311 elec | V4:192 elec — electrode imbalance corrected via per-ROI PC\n"
    f"CNN features: {LAYER_LABELS[MAIN_LAYER]}",
    fontsize=10, fontweight='bold', y=1.03)
plt.tight_layout()
save_fig(fig3, 'fig_3source_pid.pdf')
plt.show()
block_meta_plot = [
    ('V1IT_vs_V4','Block: V1+IT  vs  V4','V4 (left-out)','V1+IT block',
     '#9467bd','#8c564b',ROI_COLORS['V4']),
    ('V1V4_vs_IT','Block: V1+V4  vs  IT','IT (left-out)','V1+V4 block',
     '#e377c2','#d62728',ROI_COLORS['IT']),
    ('ITV4_vs_V1','Block: IT+V4  vs  V1','V1 (left-out)','IT+V4 block',
     '#17becf','#1f77b4',ROI_COLORS['V1']),
]
fig4, axes4 = plt.subplots(2,3,figsize=(24,15)); fig4.patch.set_facecolor('#fafafa')
for idx, (bk,title,lo_label,pair_label,pair_col,syn_col,lo_col) in enumerate(block_meta_plot):
    sig_b = (blk_syn[bk]>0) & post
    ax_top = axes4[0,idx]
    ax_top.stackplot(t_axis,
                     np.maximum(blk_red[bk],0),
                     np.maximum(blk_ulo[bk],0),
                     np.maximum(blk_upair[bk],0),
                     np.maximum(blk_syn[bk],0),
                     labels=['Redundancy',
                             f'Unique {lo_label} (per-ROI PC)',
                             f'Unique {pair_label} (joint PC of pair PCs)',
                             'Synergy (block-joint)'],
                     colors=['#7f7f7f',lo_col,pair_col,syn_col], alpha=0.85)
    ax_top.fill_between(t_axis, blk_tot[bk]*0.97, blk_tot[bk]*1.03,
                        color='k', alpha=0.08, label='±3% total (reference)')
    shade_sig(ax_top, t_axis, sig_b, color='gold', alpha=0.18)
    ax_top.set_title(title, fontsize=11, fontweight='bold')
    ax_top.set_xlabel("Time (ms)", fontsize=10)
    if idx==0: ax_top.set_ylabel("MI (nats)", fontsize=10)
    ax_top.grid(True,ls=':',lw=0.5,alpha=0.7); ax_top.set_facecolor('#fafafa')
    add_stim(ax_top, label=(idx==0))
    h, l = ax_top.get_legend_handles_labels()
    ax_top.legend(h[::-1], l[::-1], loc='upper left', frameon=False, fontsize=8)
    ax_bot = axes4[1,idx]
    ax_bot.plot(t_axis, np.maximum(blk_upair[bk],0),
                color=pair_col, lw=2.5, label=f'Unique {pair_label}')
    ax_bot.fill_between(t_axis, np.zeros_like(blk_upair[bk]),
                        np.maximum(blk_upair[bk],0), color=pair_col, alpha=0.18)
    ax_bot.plot(t_axis, np.maximum(blk_ulo[bk],0),
                color=lo_col, lw=2.5, ls='--', label=f'Unique {lo_label}')
    ax_bot.fill_between(t_axis, np.zeros_like(blk_ulo[bk]),
                        np.maximum(blk_ulo[bk],0), color=lo_col, alpha=0.18)
    ax_bot.plot(t_axis, np.maximum(blk_syn[bk],0),
                color=syn_col, lw=1.5, ls=':', alpha=0.75, label='Synergy (ref)')
    shade_sig(ax_bot, t_axis, sig_b, color='gold', alpha=0.15)
    ax_bot.set_title(f"Unique info: {pair_label}  vs  {lo_label}",
                     fontsize=10, fontweight='bold')
    ax_bot.set_xlabel("Time (ms)", fontsize=10)
    if idx==0: ax_bot.set_ylabel("Unique MI (nats)", fontsize=10)
    ax_bot.legend(loc='upper left', frameon=False, fontsize=9)
    ax_bot.grid(True,ls=':',lw=0.5,alpha=0.7); ax_bot.set_facecolor('#fafafa')
    add_stim(ax_bot, False)
fig4.suptitle(
    "Block PID: pair joint = 2nd-level MP-PCA on pair's per-ROI PCs (blind to left-out)\n"
    "Left-out = independent per-ROI PC | Synergy = emergence when left-out joins pair\n"
    "Top: full decomposition  |  Bottom: pair-block unique info vs left-out unique info\n"
    f"ALL {len(trials_all)} trials | {N_SHUFFLES} shuffles | gold = synergy > 0 post-stim\n"
    f"CNN features: {LAYER_LABELS[MAIN_LAYER]}",
    fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig4, 'fig_block_pid.pdf')
plt.show()
print("\n" + "="*60)
print(f"Pipeline v9 complete. Total time: {(time.time()-t_grand)/60:.1f} min")
print(f"All data saved to: {DRIVE_OUT}")
print(f"\nLayers analysed ({len(LAYER_NAMES_ORDERED)}): {LAYER_NAMES_ORDERED}")
print(f"Analyses per layer: geometry | MP-k | RSA | gaussianity |")
print(f"                    3-source hybrid PID | block PID (3 configs) |")
print(f"                    pairwise PID (3 pairs) | cluster+omnibus significance | summary stats")
print(f"Window-stride sweep: {MAIN_LAYER} only, 3-way PID only (unchanged)")
print(f"\nCSV structure: all tables stacked — 'Layer' + 'Layer_label' columns identify source")
print("\nFiles saved:")
for f in sorted(os.listdir(DRIVE_OUT)):
    size_kb = os.path.getsize(os.path.join(DRIVE_OUT,f))//1024
    print(f"  {f}  ({size_kb} KB)")
print("="*60)
print("\nSignificance strategy (CORRECTED — null matrices baseline-subtracted):")
print("  KEY FIX: each shuffle column's own pre-stim mean subtracted from")
print("    null_mat_raw before ALL significance testing.  This aligns the")
print("    null distribution with the baseline-subtracted observed signal,")
print("    preventing the p=1.0 / zero-clusters failure of the original code.")
print("  PRIMARY 1: Cluster-mass permutation test (temporal extent)")
print("  PRIMARY 2: Omnibus permutation test (whole post-stim window)")
print("  BH-FDR NOT used — too conservative for smooth temporal signals.")
print("="*60)

cuML available — GPU PCA enabled
Loading neural data...
  Loaded: (300, 25248, 1024)  (80.0s)
  V1:512  IT:320  V4:192
Resolving image paths...
  Valid images: 25248
  Using ALL 25248 trials (no subsampling)

Extracting CNN features (all ResNet-18 intermediate + final layers)...
  Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 222MB/s]


  batch 5/99  (140.7s elapsed)
  batch 10/99  (268.4s elapsed)
  batch 15/99  (269.0s elapsed)
  batch 20/99  (401.9s elapsed)
  batch 25/99  (534.1s elapsed)
  batch 30/99  (542.1s elapsed)
  batch 35/99  (666.0s elapsed)
  batch 40/99  (670.9s elapsed)
  batch 45/99  (790.7s elapsed)
  batch 50/99  (924.3s elapsed)
  batch 55/99  (926.1s elapsed)
  batch 60/99  (1055.5s elapsed)
  batch 65/99  (1168.1s elapsed)
  batch 70/99  (1175.2s elapsed)
  batch 75/99  (1302.4s elapsed)
  batch 80/99  (1302.6s elapsed)
  batch 85/99  (1426.7s elapsed)
  batch 90/99  (1556.0s elapsed)
  batch 95/99  (1556.2s elapsed)
  layer1_pool: raw (25248, 64) → PCA-50 (25248, 50)
  layer2_pool: raw (25248, 128) → PCA-50 (25248, 50)
  layer3_pool: raw (25248, 256) → PCA-50 (25248, 50)
  layer4_pool: raw (25248, 512) → PCA-50 (25248, 50)
  avgpool: raw (25248, 512) → PCA-50 (25248, 50)
  Feature extraction complete  (1676.7s)

LAYER: layer1_pool  |  ResNet18-Layer1 (64ch, spatially pooled)
  CNN feature shape

In [ ]:
'''import os, warnings, time
warnings.filterwarnings("ignore", category=RuntimeWarning)
import numpy as np, pandas as pd, h5py, torch
import torchvision.models as models, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from numpy.linalg import slogdet
from scipy.linalg import eigh
from scipy.stats import spearmanr, normaltest
from sklearn.decomposition import PCA
import matplotlib, matplotlib.pyplot as plt
from matplotlib.colors import Normalize
matplotlib.rcParams.update({'font.family':'serif','font.size':11,
    'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':0.8,
    'xtick.direction':'in','ytick.direction':'in','figure.dpi':150})
# ── cuML GPU PCA ──────────────────────────────────────────────────────────────
try:
    from cuml.decomposition import PCA as cuPCA
    import cupy as cp
    CUDA_ML = True
    print("cuML available — GPU PCA enabled")
except ImportError:
    CUDA_ML = False
    print("cuML not available — CPU PCA fallback")
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════════════════════════════
N_SAMPLES        = 25_000
N_BIAS_SUB       = 5_000
N_SHUFFLES       = 300
STIM_ONSET       = 100
WINDOW_SIZE      = 15
STEP_SIZE        = 5
TIME_BINS        = np.arange(0, 300-WINDOW_SIZE, STEP_SIZE)
EMERGE_GATE_NSEM = 3.0
CLUSTER_ALPHA    = 0.05
SWEEP_CONFIGS    = [(10,2),(15,4),(20,5),(30,5),(50,10)]
ROIS             = ['V1','IT','V4']
ROI_COLORS       = {'V1':'#1f77b4','IT':'#2ca02c','V4':'#ff7f0e'}
CACHE_PATH  = '/content/monkey_full_cleaned_normalized.h5'
ACTUAL_ROOT = '/content/drive/MyDrive/THINGS_images_unzipped/object_images'
DRIVE_OUT   = '/content/drive/MyDrive/neurips_v9_results'
os.makedirs(DRIVE_OUT, exist_ok=True)
# ── Load neural data ──────────────────────────────────────────────────────────
print("="*60)
print("Loading neural data...")
t0 = time.time()
with h5py.File(CACHE_PATH,'r') as hf:
    clean_data = hf['data'][:].astype(np.float64)
T_total, N_trials, N_units = clean_data.shape
print(f"  Loaded: {clean_data.shape}  ({time.time()-t0:.1f}s)")
assert N_units==1012 and T_total==300 and N_trials>=25000
roi_clean = {'V1': clean_data[:,:,0:509],
             'IT': clean_data[:,:,509:820],
             'V4': clean_data[:,:,820:]}
assert roi_clean['V1'].shape[2]==509
assert roi_clean['IT'].shape[2]==311
assert roi_clean['V4'].shape[2]==192
print(f"  V1:{roi_clean['V1'].shape[2]}  IT:{roi_clean['IT'].shape[2]}  V4:{roi_clean['V4'].shape[2]}")
# ── Metadata — use ALL valid trials ──────────────────────────────────────────
print("Resolving image paths...")
master_metadata['full_path'] = master_metadata['relative_path'].apply(
    lambda x: os.path.join(ACTUAL_ROOT, x))
valid_metadata = (master_metadata[master_metadata['full_path'].apply(os.path.exists)]
                  .copy().reset_index(drop=True))
print(f"  Valid images: {len(valid_metadata)}")
assert len(valid_metadata) >= N_SAMPLES, \
    f"Only {len(valid_metadata)} valid images but N_SAMPLES={N_SAMPLES}"
trials_all = valid_metadata['trial_idx'].values
print(f"  Using ALL {len(trials_all)} trials (no subsampling)")
# ── CNN features (ResNet-18 avgpool → PCA-50) ─────────────────────────────────
print("Extracting CNN features...")
class THINGSDs(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            with Image.open(self.paths[i]) as img:
                return self.tf(img.convert('RGB'))
        except:
            return torch.zeros(3, 224, 224)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Device: {device}")
mdl = models.resnet18(weights='DEFAULT').to(device).eval()
pool_list = []
def _hook(m, i, o): pool_list.append(o.view(o.size(0), -1).detach().cpu().numpy())
hh = mdl.avgpool.register_forward_hook(_hook)
tf_cpu = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
normalize_gpu = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
dl = DataLoader(
    THINGSDs(valid_metadata['full_path'].tolist(), tf_cpu),
    batch_size=256,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=4,
    persistent_workers=True,
)
t0 = time.time()
with torch.no_grad():
    for b_idx, b in enumerate(dl):
        b = normalize_gpu(b.to(device, non_blocking=True))
        mdl(b)
        if (b_idx+1) % 5 == 0:
            print(f"  batch {b_idx+1}/{len(dl)}  ({time.time()-t0:.1f}s elapsed)")
hh.remove()
cnn_raw   = np.vstack(pool_list).astype(np.float64)
cnn_feats = PCA(n_components=50).fit_transform(cnn_raw)
cnn_all   = cnn_feats
print(f"  CNN features: {cnn_all.shape}  ({time.time()-t0:.1f}s)")
print("="*60)
# ══════════════════════════════════════════════════════════════════════════════
#  MARCHENKO-PASTUR PCA
# ══════════════════════════════════════════════════════════════════════════════
def mp_pca(X):
    """First-level MP-PCA: raw electrodes → signal PCs."""
    X   = np.asarray(X, dtype=np.float32)
    n, p = X.shape
    Xc  = (X - X.mean(0)).astype(np.float64)
    if CUDA_ML and p > 50:
        try:
            n_comp = min(p-1, n-1, 512)
            pca_gpu = cuPCA(n_components=n_comp, random_state=0)
            pca_gpu.fit(cp.asarray(Xc.astype(np.float32)))
            lam  = cp.asnumpy(pca_gpu.explained_variance_).astype(np.float64)
            vecs = cp.asnumpy(pca_gpu.components_).T.astype(np.float64)
        except Exception:
            cov = np.cov(Xc, rowvar=False)
            la, ve = eigh(cov); lam = la[::-1]; vecs = ve[:,::-1]
    else:
        cov = np.cov(Xc, rowvar=False)
        la, ve = eigh(cov); lam = la[::-1]; vecs = ve[:,::-1]
    gamma    = p / n
    sigma2   = float(np.median(lam[lam > 0]))
    lam_plus = sigma2 * (1.0 + np.sqrt(gamma))**2
    k        = max(1, int(np.sum(lam > lam_plus)))
    X_reduced = Xc @ vecs[:, :k]
    return X_reduced, k, lam
def mp_pca_from_pcs(pc_list):
    """
    Second-level MP-PCA: concatenate per-ROI PC projections,
    then find shared signal structure via MP threshold.
    """
    X = np.hstack(pc_list).astype(np.float64)
    n, p = X.shape
    Xc = X - X.mean(0)
    cov = np.cov(Xc, rowvar=False)
    la, ve = eigh(cov); lam = la[::-1]; vecs = ve[:,::-1]
    gamma    = p / n
    sigma2   = float(np.median(lam[lam > 0]))
    lam_plus = sigma2 * (1.0 + np.sqrt(gamma))**2
    k        = max(1, int(np.sum(lam > lam_plus)))
    X_reduced = Xc @ vecs[:, :k]
    return X_reduced, k, lam
# ══════════════════════════════════════════════════════════════════════════════
#  GAUSSIAN MI
# ══════════════════════════════════════════════════════════════════════════════
def _mi_gauss(X, Y):
    X  = np.asarray(X, dtype=np.float64); Xc = X - X.mean(0)
    Y  = np.asarray(Y, dtype=np.float64); Yc = Y - Y.mean(0)
    n  = len(Xc); dx = Xc.shape[1]
    XY = np.hstack([Xc, Yc])
    C  = (XY.T @ XY) / (n - 1)
    Cx = C[:dx, :dx]; Cy = C[dx:, dx:]
    sx, ldx = slogdet(Cx)
    sy, ldy = slogdet(Cy)
    sc, ldc = slogdet(C)
    if sx<=0 or sy<=0 or sc<=0: return 0.0
    return max(0.0, 0.5*(ldx + ldy - ldc))
def _mi_gauss_batch_shuffles(X, Y, n_sh=N_SHUFFLES, rng=None):
    if rng is None: rng = np.random.default_rng()
    X  = np.asarray(X, dtype=np.float64); Xc = X - X.mean(0)
    Y  = np.asarray(Y, dtype=np.float64)
    n  = len(Xc); dx = Xc.shape[1]
    Cxx = (Xc.T @ Xc) / (n-1)
    sx, ldx = slogdet(Cxx)
    if sx <= 0: return np.zeros(n_sh)
    mis = np.zeros(n_sh)
    for sh in range(n_sh):
        Ys = Y[rng.permutation(n)]; Yc = Ys - Ys.mean(0)
        XY = np.hstack([Xc, Yc]); C = (XY.T @ XY) / (n-1)
        Cy = C[dx:, dx:]
        sy, ldy = slogdet(Cy); sc, ldc = slogdet(C)
        if sy<=0 or sc<=0: mis[sh]=0.0
        else: mis[sh] = max(0.0, 0.5*(ldx+ldy-ldc))
    return mis
# ══════════════════════════════════════════════════════════════════════════════
#  BIAS ESTIMATION
# ══════════════════════════════════════════════════════════════════════════════
def get_bias(S_list, T, n_sh=N_SHUFFLES, n_sub=N_BIAS_SUB, rng=None):
    if rng is None: rng = np.random.default_rng()
    n   = len(T)
    idx = rng.choice(n, min(n_sub,n), replace=False)
    Ss  = [S[idx] for S in S_list]; Ts = T[idx]
    bm  = np.zeros(len(S_list)); nm = np.zeros((len(S_list),n_sh))
    bt  = 0.0;                    nt = np.zeros(n_sh)
    for sh in range(n_sh):
        perm = rng.permutation(len(Ts)); Tp = Ts[perm]
        mv   = np.array([_mi_gauss(S, Tp) for S in Ss])
        mj   = _mi_gauss(np.hstack(Ss), Tp)
        bm  += mv; bt += mj; nm[:,sh] = mv; nt[sh] = mj
    return bm/n_sh, bt/n_sh, nm, nt
def get_bias_single(Z, T, n_sh=N_SHUFFLES, n_sub=N_BIAS_SUB, rng=None):
    if rng is None: rng = np.random.default_rng()
    n   = len(T)
    idx = rng.choice(n, min(n_sub,n), replace=False)
    Zs  = Z[idx]; Ts = T[idx]
    nt  = _mi_gauss_batch_shuffles(Zs, Ts, n_sh=n_sh, rng=rng)
    return float(np.mean(nt)), nt
# ══════════════════════════════════════════════════════════════════════════════
#  PID FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════
def hybrid_pid(S_perpc, Z_joint, T, bias_mis, bias_total_perpc, bias_total_joint):
    mis = np.array([_mi_gauss(S,T) for S in S_perpc]) - bias_mis
    mis = np.maximum(0.0, mis)
    mi_joint    = max(0.0, _mi_gauss(Z_joint,T) - bias_total_joint)
    red         = float(np.min(mis))
    uniques     = [max(0.0, float(m)-red) for m in mis]
    syn         = max(0.0, mi_joint - red - sum(uniques))
    total_perpc = max(0.0, _mi_gauss(np.hstack(S_perpc),T) - bias_total_perpc)
    return {'Total':total_perpc,'Redundancy':red,'Synergy':syn,
            'Uniques':uniques,'MI_joint':mi_joint}
def block_pid(Z_pair_joint, Z_lo_pc, T, bias_mis2, bias_total_2src, bias_total_block_joint):
    Z_bj  = np.hstack([Z_pair_joint, Z_lo_pc])
    mis   = np.array([_mi_gauss(S,T) for S in [Z_pair_joint, Z_lo_pc]]) - bias_mis2
    mis   = np.maximum(0.0, mis)
    mi_bj = max(0.0, _mi_gauss(Z_bj,T) - bias_total_block_joint)
    mi_2s = max(0.0, _mi_gauss(Z_bj,T) - bias_total_2src)
    red   = float(np.min(mis))
    u_pair= max(0.0, float(mis[0])-red)
    u_lo  = max(0.0, float(mis[1])-red)
    syn   = max(0.0, mi_bj - red - u_pair - u_lo)
    return {'Total':mi_2s,'Redundancy':red,'Synergy':syn,
            'Unique_pair':u_pair,'Unique_lo':u_lo,'MI_block_joint':mi_bj}
def pairwise_pid(S_A, S_B, Z_pair_joint, T, bias_mis2, bias_total_2src, bias_total_pair_joint):
    mis   = np.array([_mi_gauss(S,T) for S in [S_A,S_B]]) - bias_mis2
    mis   = np.maximum(0.0, mis)
    mi_pj = max(0.0, _mi_gauss(Z_pair_joint,T) - bias_total_pair_joint)
    mi_2s = max(0.0, _mi_gauss(np.hstack([S_A,S_B]),T) - bias_total_2src)
    red   = float(np.min(mis))
    uA    = max(0.0, float(mis[0])-red)
    uB    = max(0.0, float(mis[1])-red)
    syn   = max(0.0, mi_pj - red - uA - uB)
    return {'Total':mi_2s,'Redundancy':red,'Synergy':syn,'Unique_A':uA,'Unique_B':uB}
# ══════════════════════════════════════════════════════════════════════════════
#  GEOMETRY, RSA, GAUSSIANITY
# ══════════════════════════════════════════════════════════════════════════════
def compute_manifold_geometry(X):
    X   = np.asarray(X, dtype=np.float64); Xc = X - X.mean(0)
    cov = np.cov(Xc, rowvar=False).astype(np.float64)
    lam = eigh(cov, eigvals_only=True).astype(np.float64)
    lam = np.sort(lam)[::-1]; lam = lam[lam>1e-10]
    if len(lam)==0: return 0.0,1.0,0.0
    pr   = float((np.sum(lam)**2)/np.sum(lam**2))
    pc1f = float(lam[0]/np.sum(lam))
    sv   = np.linalg.svd(Xc,compute_uv=False).astype(np.float64); sv=sv[sv>1e-10]
    cond = float(sv[0]/sv[-1]) if len(sv)>1 else 1.0
    return pr, cond, pc1f
def get_rsa_permtest(Xn, Xc, n_sub=500, n_rep=5, n_perm=200, seed=None):
    rng = np.random.default_rng(seed)
    n   = min(len(Xn), len(Xc)); rs=[]
    for _ in range(n_rep):
        idx = rng.choice(n, n_sub, replace=False)
        A = (Xn[idx]-Xn[idx].mean(0))/(Xn[idx].std(0)+1e-8)
        B = (Xc[idx]-Xc[idx].mean(0))/(Xc[idx].std(0)+1e-8)
        rdA=1-np.corrcoef(A); rdB=1-np.corrcoef(B)
        ti,tj=np.triu_indices(n_sub,k=1)
        r,_=spearmanr(rdA[ti,tj],rdB[ti,tj]); rs.append(float(r))
    obs_r = float(np.mean(rs))
    null_rs=[]
    idx_perm=rng.choice(n,n_sub,replace=False)
    A_p=(Xn[idx_perm]-Xn[idx_perm].mean(0))/(Xn[idx_perm].std(0)+1e-8)
    rdA_p=1-np.corrcoef(A_p)
    for _ in range(n_perm):
        shuf=rng.permutation(n_sub)
        B_s=(Xc[idx_perm[shuf]]-Xc[idx_perm[shuf]].mean(0))/(Xc[idx_perm[shuf]].std(0)+1e-8)
        rdB_s=1-np.corrcoef(B_s)
        r_null,_=spearmanr(rdA_p[ti,tj],rdB_s[ti,tj]); null_rs.append(float(r_null))
    pval=(np.sum(np.array(null_rs)>=obs_r)+1)/(n_perm+1)
    return obs_r, pval
def gaussianity_check(X, label, n_test=5000):
    rng=np.random.default_rng(0); idx=rng.choice(len(X),min(n_test,len(X)),replace=False)
    pcs=PCA(n_components=5).fit_transform(X[idx]); rows=[]
    for k in range(5):
        stat,p=normaltest(pcs[:,k])
        rows.append({'ROI':label,'PC':k+1,
                     'statistic':round(float(stat),4),'p_value':round(float(p),6)})
    return pd.DataFrame(rows)
# ══════════════════════════════════════════════════════════════════════════════
#  STATISTICS
#
#  PRIMARY SIGNIFICANCE STRATEGY (replaces BH-FDR):
#
#  1. OMNIBUS TEST: Is post-stimulus mean MI significantly greater than
#     the shuffle null? Single test per metric — no multiple comparisons.
#     Null = mean of shuffle time series over post-stimulus window.
#     p = fraction of null means >= observed mean.
#
#  2. CLUSTER-MASS PERMUTATION TEST: For temporal extent inference.
#     Threshold each time series at null_mean + 2*null_std per bin.
#     Find contiguous clusters of above-threshold bins.
#     Test each cluster's summed mass against the null distribution
#     of maximum cluster masses from shuffle time series.
#     This gains power from temporal autocorrelation in the signal.
#
#  These two tests together are the primary significance evidence.
#  BH-FDR is NOT used as primary significance — it is too conservative
#  for smooth temporal signals because it treats each bin as independent.
# ══════════════════════════════════════════════════════════════════════════════
def omnibus_permtest(obs_arr, null_mat, post_mask):
    """
    Omnibus permutation test: is the post-stimulus mean of obs_arr
    significantly above the shuffle null?
    Parameters
    ----------
    obs_arr  : (n_tbins,) observed time series (baseline-subtracted)
    null_mat : (n_tbins, n_shuffles) shuffle time series
    post_mask: (n_tbins,) boolean mask for post-stimulus bins
    Returns
    -------
    obs_mean  : float — observed post-stimulus mean
    null_means: (n_shuffles,) null distribution of post-stimulus means
    pval      : float — fraction of null means >= obs_mean
    """
    obs_mean  = float(np.mean(obs_arr[post_mask]))
    null_means = np.mean(null_mat[post_mask, :], axis=0)   # (n_shuffles,)
    pval = (np.sum(null_means >= obs_mean) + 1) / (null_mat.shape[1] + 1)
    return obs_mean, null_means, float(pval)
def null_cluster_masses(null_mat, null_mean_per_bin, null_std_per_bin):
    """
    Build the null distribution of maximum cluster masses.
    For each shuffle column, threshold at null_mean + 2*null_std per bin,
    find contiguous runs, take the maximum cluster mass.
    Parameters
    ----------
    null_mat         : (n_tbins, n_shuffles)
    null_mean_per_bin: (n_tbins,) per-bin null mean
    null_std_per_bin : (n_tbins,) per-bin null std
    Returns
    -------
    masses: list of length n_shuffles — max cluster mass per shuffle
    """
    threshold = null_mean_per_bin + 2.0 * null_std_per_bin
    masses = []
    for rep in range(null_mat.shape[1]):
        s   = null_mat[:, rep]
        sig = s > threshold
        mx  = 0.0; inc = False; cur = 0.0
        for v, f in zip(s, sig):
            if f:
                cur += abs(float(v)); inc = True
            else:
                if inc: mx = max(mx, cur)
                cur = 0.0; inc = False
        if inc: mx = max(mx, cur)
        masses.append(mx)
    return masses
def cluster_mass_test(obs_arr, null_mat, null_mean_per_bin, null_std_per_bin,
                      null_cluster_mass_dist, alpha=CLUSTER_ALPHA):
    """
    Cluster-mass permutation test for a 1D time series.
    1. Threshold observed series at per-bin (null_mean + 2*null_std).
    2. Find contiguous above-threshold clusters in observed series.
    3. For each cluster, compute its mass and p-value against
       null_cluster_mass_dist (max-cluster-mass null distribution).
    4. Mark all bins belonging to significant clusters.
    Returns
    -------
    cluster_onsets   : list of bin indices where each cluster starts
    cluster_peaks    : list of bin indices of peak within each cluster
    cluster_pvals    : list of p-values for each cluster
    sig_cluster_mask : (n_tbins,) bool — True for bins in significant clusters
    null_mean_band   : (n_tbins,) null mean — for plotting
    null_std_band    : (n_tbins,) null std  — for plotting
    """
    threshold = null_mean_per_bin + 2.0 * null_std_per_bin
    # Find clusters in observed series
    clusters = []
    in_cl = False; start = 0
    for i, v in enumerate(obs_arr):
        if v > threshold[i] and not in_cl:
            in_cl = True; start = i
        elif (v <= threshold[i] or i == len(obs_arr)-1) and in_cl:
            end = i-1 if v <= threshold[i] else i
            clusters.append((start, end))
            in_cl = False
    if in_cl:
        clusters.append((start, len(obs_arr)-1))
    null_arr = np.array(null_cluster_mass_dist)
    onsets = []; peaks = []; pvals = []
    sig_mask = np.zeros(len(obs_arr), dtype=bool)
    for s, e in clusters:
        mass = float(np.sum(obs_arr[s:e+1]))
        pval = (np.sum(null_arr >= mass) + 1) / (len(null_arr) + 1)
        onsets.append(s)
        peaks.append(s + int(np.argmax(obs_arr[s:e+1])))
        pvals.append(pval)
        if pval < alpha:
            sig_mask[s:e+1] = True
    return onsets, peaks, pvals, sig_mask, null_mean_per_bin, null_std_per_bin
def compute_null_stats_per_bin(null_mat):
    """Per-bin null mean and std from the shuffle matrix."""
    return null_mat.mean(axis=1), null_mat.std(axis=1)
def add_stim(ax, label=True):
    ax.axvline(STIM_ONSET, color='black', ls='--', lw=1.8, alpha=0.7, zorder=5,
               label='Stimulus onset' if label else None)
def shade_sig(ax, ta, mask, color='gold', alpha=0.25):
    for i, s in enumerate(mask):
        if s: ax.axvspan(ta[i]-STEP_SIZE/2, ta[i]+STEP_SIZE/2,
                         color=color, alpha=alpha, zorder=1)
def shade_null_band(ax, t_axis, null_mean, null_std, n_sd=2, color='grey', alpha=0.18):
    """Plot the shuffle null mean ± n_sd*std band as visual reference."""
    upper = null_mean + n_sd * null_std
    lower = null_mean - n_sd * null_std
    ax.fill_between(t_axis, lower, upper, color=color, alpha=alpha,
                    label=f'Shuffle null ±{n_sd}SD', zorder=0)
    ax.plot(t_axis, null_mean, color=color, lw=0.8, ls=':', alpha=0.6)
def save_csv(df, fname, title=None):
    path = os.path.join(DRIVE_OUT, fname)
    df.to_csv(path, index=False)
    if title: print(f"  Saved: {fname}  ({len(df)} rows)")
def save_fig(fig, fname):
    path = os.path.join(DRIVE_OUT, fname)
    fig.savefig(path, bbox_inches='tight', dpi=150)
    print(f"  Saved: {fname}")
# ══════════════════════════════════════════════════════════════════════════════
#  MAIN ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
n_tbins = len(TIME_BINS)
rng     = np.random.default_rng(0)
cnn_all_trials = cnn_all
geom     = {r:{'PR':[],'CN':[],'PC1_Frac':[]} for r in ROIS}
rsa      = {r:{'r':[],'pval':[]} for r in ROIS}
neu2img  = []
null_total_list = [[] for _ in range(n_tbins)]
null_syn_list   = [[] for _ in range(n_tbins)]
block_pid_res   = {'V1IT_vs_V4':[],'V1V4_vs_IT':[],'ITV4_vs_V1':[]}
pair_pid_res    = {'V1_IT':[],'V1_V4':[],'IT_V4':[]}
mp_k = {r:[] for r in ROIS}
mp_k_joint = []
mp_k_block = {'V1IT_vs_V4':[],'V1V4_vs_IT':[],'ITV4_vs_V1':[]}
mp_k_pair  = {'V1_IT':[],'V1_V4':[],'IT_V4':[]}
gauss_rows = []
t_loop = time.time()
print(f"\nMain analysis: {n_tbins} time bins × {N_SHUFFLES} shuffles")
print(f"ALL {len(trials_all)} trials used for MI | N_BIAS_SUB={N_BIAS_SUB} for bias")
print(f"Architecture: per-ROI MP-PCA → concat PCs → 2nd-level MP-PCA (joint)")
print(f"Significance: Cluster-mass permutation + Omnibus permutation (no BH-FDR)")
print("="*60)
for t_idx, t in enumerate(TIME_BINS):
    raw = {}
    for roi in ROIS:
        raw[roi] = np.mean(
            roi_clean[roi][t:t+WINDOW_SIZE, trials_all, :], axis=0
        ).astype(np.float64)
    for roi in ROIS:
        pr, cond, pc1f = compute_manifold_geometry(raw[roi])
        geom[roi]['PR'].append(pr)
        geom[roi]['CN'].append(cond)
        geom[roi]['PC1_Frac'].append(pc1f)
    pc_roi = {}
    for roi in ROIS:
        pc_roi[roi], k_r, _ = mp_pca(raw[roi])
        mp_k[roi].append(k_r)
    if t_idx == 0:
        for roi in ROIS:
            df_g = gaussianity_check(pc_roi[roi], roi)
            df_g['Time_ms'] = float(t + WINDOW_SIZE/2)
            gauss_rows.append(df_g)
    for roi in ROIS:
        r_val, pval = get_rsa_permtest(
            pc_roi[roi], cnn_all_trials, n_sub=500, n_rep=5,
            n_perm=200, seed=t_idx*100)
        rsa[roi]['r'].append(r_val)
        rsa[roi]['pval'].append(pval)
    S_perpc  = [pc_roi[r] for r in ROIS]
    Z_joint3, k_j3, _ = mp_pca_from_pcs(S_perpc)
    mp_k_joint.append(k_j3)
    bm3, bt3_perpc, nm3, nt3 = get_bias(S_perpc, cnn_all_trials, rng=rng)
    bt3_joint, nt3_joint      = get_bias_single(Z_joint3, cnn_all_trials, rng=rng)
    pid3 = hybrid_pid(S_perpc, Z_joint3, cnn_all_trials, bm3, bt3_perpc, bt3_joint)
    neu2img.append(pid3)
    null_total_list[t_idx].extend(nt3.tolist())
    null_syn_vals = []
    for sh in range(N_SHUFFLES):
        nm = nm3[:,sh]; nr = float(np.min(nm))
        ns = max(0.0, nt3[sh] - np.sum([max(0., float(m)-nr) for m in nm]) - nr)
        null_syn_vals.append(ns)
    null_syn_list[t_idx].extend(null_syn_vals)
    block_configs = {
        'V1IT_vs_V4': (['V1','IT'], 'V4'),
        'V1V4_vs_IT': (['V1','V4'], 'IT'),
        'ITV4_vs_V1': (['IT','V4'], 'V1'),
    }
    roi_idx = {'V1':0, 'IT':1, 'V4':2}
    for bk, (pair_rois, lo_roi) in block_configs.items():
        Z_pair_joint, k_pair, _ = mp_pca_from_pcs([pc_roi[r] for r in pair_rois])
        Z_lo_pc = pc_roi[lo_roi]
        mp_k_block[bk].append(k_pair)
        bm2, bt2, _, _ = get_bias([Z_pair_joint, Z_lo_pc], cnn_all_trials, rng=rng)
        bt_bj, _        = get_bias_single(
            np.hstack([Z_pair_joint, Z_lo_pc]), cnn_all_trials, rng=rng)
        block_pid_res[bk].append(
            block_pid(Z_pair_joint, Z_lo_pc, cnn_all_trials, bm2, bt2, bt_bj))
    pair_configs = {'V1_IT':('V1','IT'), 'V1_V4':('V1','V4'), 'IT_V4':('IT','V4')}
    for pk, (rA, rB) in pair_configs.items():
        Z_pair2, k_p2, _ = mp_pca_from_pcs([pc_roi[rA], pc_roi[rB]])
        mp_k_pair[pk].append(k_p2)
        bias_marginals = np.array([bm3[roi_idx[rA]], bm3[roi_idx[rB]]])
        bt_2src = float(np.mean([bm3[roi_idx[rA]], bm3[roi_idx[rB]]]))
        bt_pj, _ = get_bias_single(Z_pair2, cnn_all_trials, rng=rng)
        pair_pid_res[pk].append(
            pairwise_pid(pc_roi[rA], pc_roi[rB], Z_pair2,
                         cnn_all_trials, bias_marginals, bt_2src, bt_pj))
    elapsed = time.time()-t_loop
    eta     = (elapsed/(t_idx+1))*(n_tbins-t_idx-1)
    if (t_idx+1)%5==0:
        print(f"  bin {t_idx+1}/{n_tbins}  t={t}ms  "
              f"elapsed={elapsed:.0f}s  ETA={eta:.0f}s")
print(f"\nAnalysis complete in {(time.time()-t_loop)/60:.1f} min")
print("="*60)
# ══════════════════════════════════════════════════════════════════════════════
#  AGGREGATION
# ══════════════════════════════════════════════════════════════════════════════
t_axis = TIME_BINS + WINDOW_SIZE/2.0
pre    = t_axis < STIM_ONSET
post   = t_axis >= STIM_ONSET
t_post = t_axis[post]
def _bs(arr):
    b = float(np.mean(arr[pre])) if np.any(pre) else 0.0
    return arr - b
geom_mu = {r:{k:np.array(v) for k,v in geom[r].items()} for r in ROIS}
rsa_r   = {r: _bs(np.array(rsa[r]['r']))   for r in ROIS}
rsa_pv  = {r: np.array(rsa[r]['pval'])     for r in ROIS}
rsa_sig = {r: rsa_pv[r] < 0.05            for r in ROIS}
tot_arr = _bs(np.array([p['Total']      for p in neu2img]))
red_arr = _bs(np.array([p['Redundancy'] for p in neu2img]))
syn_arr = _bs(np.array([p['Synergy']    for p in neu2img]))
uv1_arr = _bs(np.array([p['Uniques'][0] for p in neu2img]))
uit_arr = _bs(np.array([p['Uniques'][1] for p in neu2img]))
uv4_arr = _bs(np.array([p['Uniques'][2] for p in neu2img]))
mj_arr  = _bs(np.array([p['MI_joint']   for p in neu2img]))
# Build null matrices (n_tbins × n_shuffles)
null_mat_tot = np.array([null_total_list[t] for t in range(n_tbins)])
null_mat_syn = np.array([null_syn_list[t]   for t in range(n_tbins)])
# ── Per-bin null statistics (for cluster threshold) ───────────────────────────
null_mean_tot, null_std_tot = compute_null_stats_per_bin(null_mat_tot)
null_mean_syn, null_std_syn = compute_null_stats_per_bin(null_mat_syn)
# ── Omnibus permutation tests ─────────────────────────────────────────────────
omni_mean_tot, omni_null_tot, omni_pval_tot = omnibus_permtest(tot_arr, null_mat_tot, post)
omni_mean_syn, omni_null_syn, omni_pval_syn = omnibus_permtest(syn_arr, null_mat_syn, post)
print(f"\nOmnibus test — Total MI:  obs_mean={omni_mean_tot:.4f} nats  p={omni_pval_tot:.4f}")
print(f"Omnibus test — Synergy:   obs_mean={omni_mean_syn:.4f} nats  p={omni_pval_syn:.4f}")
# ── Null cluster-mass distributions ──────────────────────────────────────────
ncm_tot = null_cluster_masses(null_mat_tot, null_mean_tot, null_std_tot)
ncm_syn = null_cluster_masses(null_mat_syn, null_mean_syn, null_std_syn)
# ── Cluster-mass tests ────────────────────────────────────────────────────────
(clo_tot, clp_tot, clpv_tot,
 sig_cluster_tot, _, _) = cluster_mass_test(
    tot_arr, null_mat_tot, null_mean_tot, null_std_tot, ncm_tot)
(clo_syn, clp_syn, clpv_syn,
 sig_cluster_syn, _, _) = cluster_mass_test(
    syn_arr, null_mat_syn, null_mean_syn, null_std_syn, ncm_syn)
print(f"\nCluster-mass — Total MI: {len(clo_tot)} cluster(s), "
      f"significant: {sum(p<CLUSTER_ALPHA for p in clpv_tot)}")
print(f"Cluster-mass — Synergy:  {len(clo_syn)} cluster(s), "
      f"significant: {sum(p<CLUSTER_ALPHA for p in clpv_syn)}")
# Emergence index
def gated_emerge(syn, tot):
    pre_std = float(np.std(tot[pre])) if np.any(pre) else 1e-6
    gate    = EMERGE_GATE_NSEM * pre_std
    valid   = tot > gate
    em      = np.where(valid, syn/np.maximum(tot,1e-12), np.nan)
    return em
em_arr = gated_emerge(syn_arr, tot_arr)
def blk(bk, c): return _bs(np.array([p[c] for p in block_pid_res[bk]]))
blk_tot  = {bk:blk(bk,'Total')        for bk in block_pid_res}
blk_red  = {bk:blk(bk,'Redundancy')   for bk in block_pid_res}
blk_syn  = {bk:blk(bk,'Synergy')      for bk in block_pid_res}
blk_upair= {bk:blk(bk,'Unique_pair')  for bk in block_pid_res}
blk_ulo  = {bk:blk(bk,'Unique_lo')    for bk in block_pid_res}
def pw(pk, c): return _bs(np.array([p[c] for p in pair_pid_res[pk]]))
pair_tot = {pk:pw(pk,'Total')      for pk in pair_pid_res}
pair_red = {pk:pw(pk,'Redundancy') for pk in pair_pid_res}
pair_syn = {pk:pw(pk,'Synergy')    for pk in pair_pid_res}
pair_uA  = {pk:pw(pk,'Unique_A')   for pk in pair_pid_res}
pair_uB  = {pk:pw(pk,'Unique_B')   for pk in pair_pid_res}
print("\nAggregation complete. Saving all data to Drive...")
# ══════════════════════════════════════════════════════════════════════════════
#  SAVE ALL DATA TO DRIVE
# ══════════════════════════════════════════════════════════════════════════════
# 1. Geometry
rows=[]
for roi in ROIS:
    for ti,tv in enumerate(t_axis):
        rows.append({'ROI':roi,'Time_ms':round(float(tv),1),
                     'PR':round(float(geom_mu[roi]['PR'][ti]),6),
                     'CN':round(float(geom_mu[roi]['CN'][ti]),6),
                     'PC1_Frac':round(float(geom_mu[roi]['PC1_Frac'][ti]),6)})
save_csv(pd.DataFrame(rows), 'table_geometry.csv', 'Geometry')
# 2. MP-k dimensionality
rows=[]
for roi in ROIS:
    for ti,tv in enumerate(t_axis):
        rows.append({'Source':roi,'Time_ms':round(float(tv),1),
                     'k':int(mp_k[roi][ti])})
for ti,tv in enumerate(t_axis):
    rows.append({'Source':'Joint_all3','Time_ms':round(float(tv),1),
                 'k':int(mp_k_joint[ti])})
for bk in block_pid_res:
    for ti,tv in enumerate(t_axis):
        rows.append({'Source':f'Block_{bk}','Time_ms':round(float(tv),1),
                     'k':int(mp_k_block[bk][ti])})
for pk in pair_pid_res:
    for ti,tv in enumerate(t_axis):
        rows.append({'Source':f'Pair_{pk}','Time_ms':round(float(tv),1),
                     'k':int(mp_k_pair[pk][ti])})
save_csv(pd.DataFrame(rows), 'table_mp_dimensionality.csv', 'MP-k')
# 3. RSA
rows=[]
for roi in ROIS:
    for ti,tv in enumerate(t_axis):
        rows.append({'ROI':roi,'Time_ms':round(float(tv),1),
                     'Spearman_r':round(float(rsa_r[roi][ti]),6),
                     'pval_perm':round(float(rsa_pv[roi][ti]),6),
                     'Sig_p05':bool(rsa_sig[roi][ti])})
save_csv(pd.DataFrame(rows), 'table_rsa.csv', 'RSA')
# 4. 3-source Hybrid PID — full time series with cluster-mass + omnibus significance
# Per-bin null mean and std are also saved so reviewers can reconstruct the threshold
rows=[]
for ti,tv in enumerate(t_axis):
    rows.append({
        'Time_ms':          round(float(tv),1),
        'Total':            round(float(tot_arr[ti]),6),
        'Redundancy':       round(float(red_arr[ti]),6),
        'Synergy':          round(float(syn_arr[ti]),6),
        'UniqueV1':         round(float(uv1_arr[ti]),6),
        'UniqueIT':         round(float(uit_arr[ti]),6),
        'UniqueV4':         round(float(uv4_arr[ti]),6),
        'MI_joint':         round(float(mj_arr[ti]),6),
        'EmergenceIndex':   float('nan') if np.isnan(em_arr[ti])
                            else round(float(em_arr[ti]),6),
        # Null statistics per bin (shuffle null mean and std)
        'null_mean_Total':  round(float(null_mean_tot[ti]),6),
        'null_std_Total':   round(float(null_std_tot[ti]),6),
        'null_mean_Syn':    round(float(null_mean_syn[ti]),6),
        'null_std_Syn':     round(float(null_std_syn[ti]),6),
        # Cluster-mass significance: True if bin belongs to a significant cluster
        'sig_cluster_Total':bool(sig_cluster_tot[ti]),
        'sig_cluster_Syn':  bool(sig_cluster_syn[ti]),
        # Omnibus p-value — same for all bins (whole-window test)
        'omnibus_pval_Total': round(float(omni_pval_tot),6),
        'omnibus_pval_Syn':   round(float(omni_pval_syn),6),
        'omnibus_mean_Total': round(float(omni_mean_tot),6),
        'omnibus_mean_Syn':   round(float(omni_mean_syn),6),
    })
save_csv(pd.DataFrame(rows), 'table_3source_pid.csv', '3-source PID')
# 5. Cluster statistics — all clusters with onset, peak, mass, p-value
rows=[]
for k,(onset,peak,pval) in enumerate(zip(clo_tot,clp_tot,clpv_tot)):
    rows.append({
        'Metric':        'TotalMI',
        'Cluster':       k,
        'Onset_ms':      round(float(t_axis[onset]),1),
        'Peak_ms':       round(float(t_axis[peak]),1),
        'Peak_val':      round(float(tot_arr[peak]),6),
        'Cluster_mass':  round(float(np.sum(tot_arr[onset:peak+1])),6),
        'Cluster_pval':  round(float(pval),6),
        'Significant':   bool(pval < CLUSTER_ALPHA),
    })
for k,(onset,peak,pval) in enumerate(zip(clo_syn,clp_syn,clpv_syn)):
    rows.append({
        'Metric':        'Synergy',
        'Cluster':       k,
        'Onset_ms':      round(float(t_axis[onset]),1),
        'Peak_ms':       round(float(t_axis[peak]),1),
        'Peak_val':      round(float(syn_arr[peak]),6),
        'Cluster_mass':  round(float(np.sum(syn_arr[onset:peak+1])),6),
        'Cluster_pval':  round(float(pval),6),
        'Significant':   bool(pval < CLUSTER_ALPHA),
    })
# Also save omnibus results as summary rows
rows.append({
    'Metric':'TotalMI_omnibus','Cluster':0,
    'Onset_ms':float('nan'),'Peak_ms':float('nan'),
    'Peak_val':round(omni_mean_tot,6),'Cluster_mass':float('nan'),
    'Cluster_pval':round(omni_pval_tot,6),'Significant':bool(omni_pval_tot<CLUSTER_ALPHA),
})
rows.append({
    'Metric':'Synergy_omnibus','Cluster':0,
    'Onset_ms':float('nan'),'Peak_ms':float('nan'),
    'Peak_val':round(omni_mean_syn,6),'Cluster_mass':float('nan'),
    'Cluster_pval':round(omni_pval_syn,6),'Significant':bool(omni_pval_syn<CLUSTER_ALPHA),
})
save_csv(pd.DataFrame(rows), 'table_cluster_stats.csv', 'Cluster + omnibus stats')
# 6. Block PID
block_meta_save = [
    ('V1IT_vs_V4','V1+IT vs V4','V4','V1+IT'),
    ('V1V4_vs_IT','V1+V4 vs IT','IT','V1+V4'),
    ('ITV4_vs_V1','IT+V4 vs V1','V1','IT+V4'),
]
rows=[]
for bk,title,lo_label,pair_label in block_meta_save:
    for ti,tv in enumerate(t_axis):
        rows.append({'Config':title,'Time_ms':round(float(tv),1),
                     'Total':round(float(blk_tot[bk][ti]),6),
                     'Redundancy':round(float(blk_red[bk][ti]),6),
                     f'Unique_{pair_label}':round(float(blk_upair[bk][ti]),6),
                     f'Unique_{lo_label}':round(float(blk_ulo[bk][ti]),6),
                     'Synergy':round(float(blk_syn[bk][ti]),6)})
save_csv(pd.DataFrame(rows), 'table_block_pid.csv', 'Block PID')
# 7. Pairwise PID
pair_meta_save = [
    ('V1_IT','V1 vs IT','V1','IT'),
    ('V1_V4','V1 vs V4','V1','V4'),
    ('IT_V4','IT vs V4','IT','V4'),
]
rows=[]
for pk,label,lA,lB in pair_meta_save:
    for ti,tv in enumerate(t_axis):
        rows.append({'Pair':label,'Time_ms':round(float(tv),1),
                     'Total':round(float(pair_tot[pk][ti]),6),
                     'Redundancy':round(float(pair_red[pk][ti]),6),
                     f'Unique_{lA}':round(float(pair_uA[pk][ti]),6),
                     f'Unique_{lB}':round(float(pair_uB[pk][ti]),6),
                     'Synergy':round(float(pair_syn[pk][ti]),6)})
save_csv(pd.DataFrame(rows), 'table_pairwise_pid.csv', 'Pairwise PID')
# 8. Gaussianity
if gauss_rows:
    gauss_df = pd.concat(gauss_rows, ignore_index=True)
    save_csv(gauss_df, 'table_gaussianity.csv', 'Gaussianity')
# 9. Window-stride robustness sweep
print("\nRunning window-stride robustness sweep...")
rng_sw    = np.random.default_rng(42)
SWEEP_N   = 5000
idx_sw    = rng_sw.choice(len(trials_all), SWEEP_N, replace=False)
trials_sw = trials_all[idx_sw]
cnn_sw    = cnn_all_trials[idx_sw]
sweep_rows=[]
for ws,ss in SWEEP_CONFIGS:
    print(f"  W{ws}/S{ss}...")
    tbins_sw=np.arange(0,300-ws,ss); taxis_sw=tbins_sw+ws/2.0
    pre_sw=taxis_sw<STIM_ONSET; post_sw=taxis_sw>=STIM_ONSET
    tot_sw=[]; syn_sw=[]; rsa_sw=[]
    for t in tbins_sw:
        raw_sw={roi:np.mean(roi_clean[roi][t:t+ws,trials_sw,:],axis=0).astype(np.float64)
                for roi in ROIS}
        pc_sw={roi:mp_pca(raw_sw[roi])[0] for roi in ROIS}
        Zj_sw,_,_=mp_pca_from_pcs([pc_sw[r] for r in ROIS])
        S_sw=[pc_sw[r] for r in ROIS]
        bm_sw,bt_sw,_,_=get_bias(S_sw,cnn_sw,n_sh=10,n_sub=1000,rng=rng_sw)
        bt_j_sw,_=get_bias_single(Zj_sw,cnn_sw,n_sh=10,n_sub=1000,rng=rng_sw)
        pid_sw=hybrid_pid(S_sw,Zj_sw,cnn_sw,bm_sw,bt_sw,bt_j_sw)
        tot_sw.append(pid_sw['Total']); syn_sw.append(pid_sw['Synergy'])
        rsa_sw.append(get_rsa_permtest(pc_sw['IT'],cnn_sw,n_sub=300,n_rep=3,n_perm=50)[0])
    tot_sw=np.array(tot_sw); syn_sw=np.array(syn_sw); rsa_sw=np.array(rsa_sw)
    for arr in [tot_sw,syn_sw,rsa_sw]:
        b=float(np.mean(arr[pre_sw])) if np.any(pre_sw) else 0.0; arr-=b
    def _pk(a,m):   return float(np.max(a[m])) if np.any(m) else float('nan')
    def _lat(a,pr,po,tax):
        ps=float(np.std(a[pr])) if np.any(pr) else 1e-9
        c=np.where((a>2*ps)&po)[0]
        return round(float(tax[c[0]]),1) if len(c)>0 else float('nan')
    def _pkl(a,po,tax):
        pi=np.where(po)[0]
        return round(float(tax[pi[np.argmax(a[pi])]]),1) if len(pi)>0 else float('nan')
    sweep_rows.append({'Window_ms':ws,'Stride_ms':ss,'N_bins':len(tbins_sw),
                       'Peak_TotalMI':round(_pk(tot_sw,post_sw),6),
                       'Peak_Synergy':round(_pk(syn_sw,post_sw),6),
                       'Peak_RSA_IT': round(_pk(rsa_sw,post_sw),6),
                       'Onset_TotalMI_ms':_lat(tot_sw,pre_sw,post_sw,taxis_sw),
                       'Onset_Synergy_ms':_lat(syn_sw,pre_sw,post_sw,taxis_sw),
                       'Peak_TotalMI_ms': _pkl(tot_sw,post_sw,taxis_sw),
                       'Peak_Synergy_ms': _pkl(syn_sw,post_sw,taxis_sw)})
    print(f"    peak_tot={sweep_rows[-1]['Peak_TotalMI']:.4f}  "
          f"peak_syn={sweep_rows[-1]['Peak_Synergy']:.4f}")
save_csv(pd.DataFrame(sweep_rows), 'table_window_stride_robustness.csv', 'Sweep')
# 10. Global summary statistics
sum_rows=[]
def _pk_s(arr,label):
    p=arr[post]; i=int(np.argmax(p))
    c=np.where(arr[post]>0)[0]
    onset=round(float(t_post[c[0]]),1) if len(c)>0 else float('nan')
    return {'Metric':label,'Peak_val':round(float(p[i]),6),
            'Peak_ms':round(float(t_post[i]),1),'Onset_ms':onset}
def _tr_s(arr,label):
    p=arr[post]; i=int(np.argmin(p))
    return {'Metric':label,'Trough_val':round(float(p[i]),6),
            'Trough_ms':round(float(t_post[i]),1)}
for roi in ROIS:
    sum_rows.append(_tr_s(geom_mu[roi]['PR'], f'PR_trough_{roi}'))
    sum_rows.append(_pk_s(geom_mu[roi]['PR'], f'PR_peak_{roi}'))
    sum_rows.append(_pk_s(geom_mu[roi]['CN'], f'CN_peak_{roi}'))
    sum_rows.append(_pk_s(rsa_r[roi],         f'RSA_peak_{roi}'))
    sig_bins=np.where(rsa_sig[roi] & post)[0]
    rsa_onset=round(float(t_post[sig_bins[0]]),1) if len(sig_bins)>0 else float('nan')
    sum_rows.append({'Metric':f'RSA_onset_{roi}','Onset_ms':rsa_onset,
                     'Peak_val':float('nan'),'Peak_ms':float('nan'),
                     'Trough_val':float('nan'),'Trough_ms':float('nan')})
sum_rows.append(_pk_s(tot_arr,'TotalMI_peak'))
sum_rows.append(_pk_s(syn_arr,'Synergy_peak'))
# Cluster-mass onset = onset of first significant cluster
for metric,arr,clo,clpv in [('TotalMI',tot_arr,clo_tot,clpv_tot),
                              ('Synergy',syn_arr,clo_syn,clpv_syn)]:
    sig_cl = [(o,p) for o,p in zip(clo,clpv) if p<CLUSTER_ALPHA]
    onset  = round(float(t_axis[sig_cl[0][0]]),1) if sig_cl else float('nan')
    n_sig  = int(sum(p<CLUSTER_ALPHA for p in clpv))
    sum_rows.append({
        'Metric':           f'{metric}_cluster_onset',
        'Onset_ms':         onset,
        'N_sig_clusters':   n_sig,
        'omnibus_pval':     round(omni_pval_tot if metric=='TotalMI' else omni_pval_syn, 6),
        'Peak_val':         float('nan'),'Peak_ms':float('nan'),
        'Trough_val':       float('nan'),'Trough_ms':float('nan'),
    })
for k,(onset,peak,pval) in enumerate(zip(clo_tot,clp_tot,clpv_tot)):
    sum_rows.append({'Metric':f'TotalMI_cluster{k}',
                     'Onset_ms':round(float(t_axis[onset]),1),
                     'Peak_ms':round(float(t_axis[peak]),1),
                     'Peak_val':round(float(tot_arr[peak]),6),
                     'Cluster_pval':round(float(pval),6),
                     'Significant':bool(pval<CLUSTER_ALPHA)})
for k,(onset,peak,pval) in enumerate(zip(clo_syn,clp_syn,clpv_syn)):
    sum_rows.append({'Metric':f'Synergy_cluster{k}',
                     'Onset_ms':round(float(t_axis[onset]),1),
                     'Peak_ms':round(float(t_axis[peak]),1),
                     'Peak_val':round(float(syn_arr[peak]),6),
                     'Cluster_pval':round(float(pval),6),
                     'Significant':bool(pval<CLUSTER_ALPHA)})
# Omnibus results in summary
sum_rows.append({'Metric':'Omnibus_TotalMI',
                 'omnibus_obs_mean':round(omni_mean_tot,6),
                 'omnibus_pval':round(omni_pval_tot,6),
                 'Significant':bool(omni_pval_tot<CLUSTER_ALPHA)})
sum_rows.append({'Metric':'Omnibus_Synergy',
                 'omnibus_obs_mean':round(omni_mean_syn,6),
                 'omnibus_pval':round(omni_pval_syn,6),
                 'Significant':bool(omni_pval_syn<CLUSTER_ALPHA)})
for bk,title,lo_label,pair_label in block_meta_save:
    sum_rows.append(_pk_s(blk_syn[bk],   f'BlockSyn_peak_{bk}'))
    sum_rows.append(_pk_s(blk_upair[bk], f'BlockUniquePair_peak_{bk}'))
    sum_rows.append(_pk_s(blk_ulo[bk],   f'BlockUniqueLO_peak_{bk}'))
for pk,label,*_ in pair_meta_save:
    sum_rows.append(_pk_s(pair_syn[pk], f'PairSyn_peak_{pk}'))
    c=np.where((pair_syn[pk]>0)&post)[0]
    onset=round(float(t_post[c[0]]),1) if len(c)>0 else float('nan')
    sum_rows.append({'Metric':f'PairSyn_onset_{pk}','Onset_ms':onset,
                     'Peak_val':float('nan'),'Peak_ms':float('nan'),
                     'Trough_val':float('nan'),'Trough_ms':float('nan')})
tbl_sum=pd.DataFrame(sum_rows)
save_csv(tbl_sum, 'table_summary_statistics.csv', 'Global summary')
# ══════════════════════════════════════════════════════════════════════════════
#  DISPLAY PLOTS: 3-source PID + Block PID only
#  Significance shown as:
#   - Gold shading: bins belonging to significant clusters (cluster-mass test)
#   - Grey band: shuffle null mean ± 2SD (visual reference — NOT the threshold)
#   - Cluster annotations: p-values from cluster-mass test
# ══════════════════════════════════════════════════════════════════════════════
print("\nGenerating main display plots...")
# ── PLOT 1: 3-source Hybrid PID ───────────────────────────────────────────────
fig3,axes3=plt.subplots(1,3,figsize=(24,8)); fig3.patch.set_facecolor('#fafafa')
ax=axes3[0]
ax.stackplot(t_axis,
             np.maximum(red_arr,0), np.maximum(uv1_arr,0),
             np.maximum(uit_arr,0), np.maximum(uv4_arr,0), np.maximum(syn_arr,0),
             labels=['Redundancy','Unique V1 (per-ROI PC)',
                     'Unique IT (per-ROI PC)','Unique V4 (per-ROI PC)',
                     'Synergy (joint PC of per-ROI PCs)'],
             colors=['#7f7f7f',ROI_COLORS['V1'],ROI_COLORS['IT'],
                     ROI_COLORS['V4'],'#d62728'], alpha=0.85)
shade_sig(ax, t_axis, sig_cluster_tot, color='gold', alpha=0.18)
ax.set_title("3-source Hybrid PID: Neu→CNN\n"
             "Joint = 2nd-level MP-PCA on per-ROI PCs\n"
             "(electrode-count-balanced inter-area structure)",
             fontsize=10, fontweight='bold')
ax.set_ylabel("MI (nats, baseline-subtracted)"); ax.set_xlabel("Time (ms)")
ax.legend(loc='upper left', fontsize='x-small', frameon=False)
add_stim(ax, False); ax.set_facecolor('#fafafa'); ax.grid(True,ls=':',lw=0.5,alpha=0.5)
ax2=axes3[1]
# Plot shuffle null band as visual reference
shade_null_band(ax2, t_axis, null_mean_tot, null_std_tot, n_sd=2,
                color='grey', alpha=0.20)
ax2.plot(t_axis, tot_arr, 'k-', lw=2.5, label='Total MI')
shade_sig(ax2, t_axis, sig_cluster_tot, color='gold', alpha=0.25)
for onset,peak,pval in zip(clo_tot,clp_tot,clpv_tot):
    if pval < CLUSTER_ALPHA:
        ax2.annotate(f'cluster p={pval:.3f}',
                     xy=(t_axis[peak], tot_arr[peak]),
                     xytext=(t_axis[peak], tot_arr[peak]*1.12),
                     arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5),
                     fontsize=9, color='darkred', ha='center')
ax2.plot(t_axis, syn_arr, color='#d62728', lw=2.0, ls='--', label='Synergy')
for onset,peak,pval in zip(clo_syn,clp_syn,clpv_syn):
    if pval < CLUSTER_ALPHA:
        ax2.annotate(f'cluster p={pval:.3f}',
                     xy=(t_axis[peak], syn_arr[peak]),
                     xytext=(t_axis[peak], syn_arr[peak]*1.25),
                     arrowprops=dict(arrowstyle='->', color='#8b0000', lw=1.5),
                     fontsize=9, color='#8b0000', ha='center')
ax2.set_title(
    f"Total MI + Synergy\n"
    f"Cluster-mass permutation (gold) | Omnibus: TotMI p={omni_pval_tot:.3f}, "
    f"Syn p={omni_pval_syn:.3f}",
    fontsize=9, fontweight='bold')
ax2.set_ylabel("MI (nats, baseline-sub)"); ax2.set_xlabel("Time (ms)")
add_stim(ax2); ax2.legend(frameon=False, fontsize=9)
ax2.set_facecolor('#fafafa'); ax2.grid(True,ls=':',lw=0.5,alpha=0.5)
ax3=axes3[2]
valid_em=~np.isnan(em_arr)
if np.any(valid_em):
    tv2=t_axis[valid_em]; ev=em_arr[valid_em]
    ax3.plot(tv2, ev, 'r-', lw=2.2, label='Gated Emergence Index')
    ax3.fill_between(tv2, np.zeros_like(ev), ev, color='r', alpha=0.15)
ax3.set_title(f"Gated Emergence Index (syn/total)\n"
              f"[NaN where total<{EMERGE_GATE_NSEM}×pre-stim noise]",
              fontsize=10, fontweight='bold')
ax3.set_ylabel("Synergy / Total MI"); ax3.set_xlabel("Time (ms)")
add_stim(ax3, False); ax3.legend(frameon=False)
ax3.set_facecolor('#fafafa'); ax3.grid(True,ls=':',lw=0.5,alpha=0.5)
fig3.suptitle(
    "3-source Hybrid PID: per-ROI MP-PCA (unique subspaces) + "
    "2nd-level MP-PCA on concatenated PCs (joint/synergy)\n"
    f"ALL {len(trials_all)} trials | {N_SHUFFLES} shuffles | "
    f"Significance: cluster-mass permutation + omnibus permutation | "
    f"bias subsampled N={N_BIAS_SUB}\n"
    "V1:509 elec | IT:311 elec | V4:192 elec — electrode imbalance corrected via per-ROI PC",
    fontsize=10, fontweight='bold', y=1.03)
plt.tight_layout()
save_fig(fig3, 'fig_3source_pid.pdf')
plt.show()
# ── PLOT 2: Block PID ─────────────────────────────────────────────────────────
block_meta_plot = [
    ('V1IT_vs_V4','Block: V1+IT  vs  V4','V4 (left-out)','V1+IT block',
     '#9467bd','#8c564b',ROI_COLORS['V4']),
    ('V1V4_vs_IT','Block: V1+V4  vs  IT','IT (left-out)','V1+V4 block',
     '#e377c2','#d62728',ROI_COLORS['IT']),
    ('ITV4_vs_V1','Block: IT+V4  vs  V1','V1 (left-out)','IT+V4 block',
     '#17becf','#1f77b4',ROI_COLORS['V1']),
]
fig4,axes4=plt.subplots(2,3,figsize=(24,15)); fig4.patch.set_facecolor('#fafafa')
for idx,(bk,title,lo_label,pair_label,pair_col,syn_col,lo_col) in enumerate(block_meta_plot):
    sig_b = (blk_syn[bk]>0) & post
    ax_top=axes4[0,idx]
    ax_top.stackplot(t_axis,
                     np.maximum(blk_red[bk],0),
                     np.maximum(blk_ulo[bk],0),
                     np.maximum(blk_upair[bk],0),
                     np.maximum(blk_syn[bk],0),
                     labels=['Redundancy',
                             f'Unique {lo_label} (per-ROI PC)',
                             f'Unique {pair_label} (joint PC of pair PCs)',
                             'Synergy (block-joint)'],
                     colors=['#7f7f7f',lo_col,pair_col,syn_col], alpha=0.85)
    ax_top.fill_between(t_axis, blk_tot[bk]*0.97, blk_tot[bk]*1.03,
                        color='k', alpha=0.08, label='±3% total (reference)')
    shade_sig(ax_top, t_axis, sig_b, color='gold', alpha=0.18)
    ax_top.set_title(title, fontsize=11, fontweight='bold')
    ax_top.set_xlabel("Time (ms)", fontsize=10)
    if idx==0: ax_top.set_ylabel("MI (nats)", fontsize=10)
    ax_top.grid(True,ls=':',lw=0.5,alpha=0.7); ax_top.set_facecolor('#fafafa')
    add_stim(ax_top, label=(idx==0))
    h,l=ax_top.get_legend_handles_labels()
    ax_top.legend(h[::-1],l[::-1],loc='upper left',frameon=False,fontsize=8)
    ax_bot=axes4[1,idx]
    ax_bot.plot(t_axis, np.maximum(blk_upair[bk],0),
                color=pair_col, lw=2.5, label=f'Unique {pair_label}')
    ax_bot.fill_between(t_axis, np.zeros_like(blk_upair[bk]),
                        np.maximum(blk_upair[bk],0), color=pair_col, alpha=0.18)
    ax_bot.plot(t_axis, np.maximum(blk_ulo[bk],0),
                color=lo_col, lw=2.5, ls='--', label=f'Unique {lo_label}')
    ax_bot.fill_between(t_axis, np.zeros_like(blk_ulo[bk]),
                        np.maximum(blk_ulo[bk],0), color=lo_col, alpha=0.18)
    ax_bot.plot(t_axis, np.maximum(blk_syn[bk],0),
                color=syn_col, lw=1.5, ls=':', alpha=0.75, label='Synergy (ref)')
    shade_sig(ax_bot, t_axis, sig_b, color='gold', alpha=0.15)
    ax_bot.set_title(f"Unique info: {pair_label}  vs  {lo_label}",
                     fontsize=10, fontweight='bold')
    ax_bot.set_xlabel("Time (ms)", fontsize=10)
    if idx==0: ax_bot.set_ylabel("Unique MI (nats)", fontsize=10)
    ax_bot.legend(loc='upper left', frameon=False, fontsize=9)
    ax_bot.grid(True,ls=':',lw=0.5,alpha=0.7); ax_bot.set_facecolor('#fafafa')
    add_stim(ax_bot, False)
fig4.suptitle(
    "Block PID: pair joint = 2nd-level MP-PCA on pair's per-ROI PCs (blind to left-out)\n"
    "Left-out = independent per-ROI PC | Synergy = emergence when left-out joins pair\n"
    "Top: full decomposition  |  Bottom: pair-block unique info vs left-out unique info\n"
    f"ALL {len(trials_all)} trials | {N_SHUFFLES} shuffles | gold = synergy > 0 post-stim",
    fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig4, 'fig_block_pid.pdf')
plt.show()
print("\n" + "="*60)
print(f"Pipeline v9 complete. Total time: {(time.time()-t_loop)/60:.1f} min")
print(f"All data saved to: {DRIVE_OUT}")
print("\nFiles saved:")
for f in sorted(os.listdir(DRIVE_OUT)):
    size_kb = os.path.getsize(os.path.join(DRIVE_OUT,f))//1024
    print(f"  {f}  ({size_kb} KB)")
print("="*60)
print("\nSignificance strategy (replaces BH-FDR):")
print("  PRIMARY 1: Cluster-mass permutation test")
print("    Threshold: per-bin shuffle null mean + 2*null_std")
print("    Null: max cluster mass across 500 shuffle time series")
print("    Bins in significant clusters marked gold in plots + sig_cluster_* in CSV")
print("  PRIMARY 2: Omnibus permutation test")
print(f"    Total MI: obs_mean={omni_mean_tot:.4f} nats  p={omni_pval_tot:.4f}")
print(f"    Synergy:  obs_mean={omni_mean_syn:.4f} nats  p={omni_pval_syn:.4f}")
print("    Null: mean of 500 shuffle post-stimulus time series")
print("  VISUAL: Shuffle null mean ± 2SD band on time series plots")
print("  NOTE: BH-FDR removed — too conservative for smooth temporal signals")
print("        (independence assumption violated; each bin treated as independent)")
print("="*60)'''

cuML available — GPU PCA enabled
Loading neural data...
  Loaded: (300, 25248, 1012)  (55.4s)
  V1:509  IT:311  V4:192
Resolving image paths...
  Valid images: 25248
  Using ALL 25248 trials (no subsampling)
Extracting CNN features...
  Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 215MB/s]


  batch 5/99  (139.3s elapsed)
  batch 10/99  (268.0s elapsed)
  batch 15/99  (268.5s elapsed)
  batch 20/99  (391.1s elapsed)
  batch 25/99  (518.0s elapsed)
  batch 30/99  (525.6s elapsed)
  batch 35/99  (649.1s elapsed)
  batch 40/99  (649.3s elapsed)
  batch 45/99  (773.2s elapsed)
  batch 50/99  (901.8s elapsed)
  batch 55/99  (902.0s elapsed)
  batch 60/99  (1031.6s elapsed)
  batch 65/99  (1141.1s elapsed)
  batch 70/99  (1151.3s elapsed)
  batch 75/99  (1279.2s elapsed)
  batch 80/99  (1279.4s elapsed)
  batch 85/99  (1403.3s elapsed)
  batch 90/99  (1533.0s elapsed)
  batch 95/99  (1533.2s elapsed)
  CNN features: (25248, 50)  (1653.0s)

Main analysis: 57 time bins × 300 shuffles
ALL 25248 trials used for MI | N_BIAS_SUB=5000 for bias
Architecture: per-ROI MP-PCA → concat PCs → 2nd-level MP-PCA (joint)
Significance: Cluster-mass permutation + Omnibus permutation (no BH-FDR)
  bin 5/57  t=20ms  elapsed=517s  ETA=5380s
  bin 10/57  t=45ms  elapsed=1043s  ETA=4901s
  bin 15/57  t

KeyboardInterrupt: 

In [ ]:
'''
# ══════════════════════════════════════════════════════════════════════════════
#  FIX: Baseline-subtract the null matrices to match the observed signal scale
#  Run this AFTER the main loop, BEFORE the significance tests
#  The null_mat_tot and null_mat_syn are on raw shuffle scale (positive, ~0.5-1.5 nats)
#  The observed tot_arr and syn_arr are baseline-subtracted (start at zero)
#  This mismatch caused p=1.0 and zero clusters — fix: subtract each shuffle
#  column's own pre-stimulus mean, exactly mirroring what _bs() does to observed
# ══════════════════════════════════════════════════════════════════════════════
print("Fixing null matrix scale (baseline-subtracting each shuffle column)...")
# Build raw null matrices from the lists collected in the main loop
null_mat_tot_raw = np.array([null_total_list[t] for t in range(n_tbins)])  # (n_tbins, n_shuffles)
null_mat_syn_raw = np.array([null_syn_list[t]   for t in range(n_tbins)])  # (n_tbins, n_shuffles)
# Baseline-subtract each column (each shuffle replicate) using its own pre-stimulus mean
# pre mask is already defined: pre = t_axis < STIM_ONSET
pre_mean_tot = null_mat_tot_raw[pre, :].mean(axis=0)  # (n_shuffles,) — one mean per shuffle
pre_mean_syn = null_mat_syn_raw[pre, :].mean(axis=0)
null_mat_tot = null_mat_tot_raw - pre_mean_tot[np.newaxis, :]  # broadcast: subtract per column
null_mat_syn = null_mat_syn_raw - pre_mean_syn[np.newaxis, :]
# Sanity check: pre-stimulus null mean should now be ~0
print(f"  null_mat_tot pre-stim mean after fix: {null_mat_tot[pre,:].mean():.6f}  (should be ~0)")
print(f"  null_mat_syn pre-stim mean after fix: {null_mat_syn[pre,:].mean():.6f}  (should be ~0)")
print(f"  null_mat_tot post-stim mean: {null_mat_tot[post,:].mean():.4f} nats")
print(f"  null_mat_syn post-stim mean: {null_mat_syn[post,:].mean():.4f} nats")
print(f"  obs tot_arr  post-stim mean: {tot_arr[post].mean():.4f} nats")
print(f"  obs syn_arr  post-stim mean: {syn_arr[post].mean():.4f} nats")
# ── Per-bin null statistics on the CORRECTED matrices ─────────────────────────
null_mean_tot, null_std_tot = null_mat_tot.mean(axis=1), null_mat_tot.std(axis=1)
null_mean_syn, null_std_syn = null_mat_syn.mean(axis=1), null_mat_syn.std(axis=1)
print(f"\n  Threshold (null_mean + 2*null_std) at t=150ms:")
bin_150 = np.argmin(np.abs(t_axis - 150))
print(f"    Total MI threshold: {null_mean_tot[bin_150] + 2*null_std_tot[bin_150]:.4f}")
print(f"    Total MI observed:  {tot_arr[bin_150]:.4f}")
print(f"    Synergy  threshold: {null_mean_syn[bin_150] + 2*null_std_syn[bin_150]:.4f}")
print(f"    Synergy  observed:  {syn_arr[bin_150]:.4f}")
# ── Omnibus tests on corrected null ───────────────────────────────────────────
omni_mean_tot, omni_null_tot, omni_pval_tot = omnibus_permtest(tot_arr, null_mat_tot, post)
omni_mean_syn, omni_null_syn, omni_pval_syn = omnibus_permtest(syn_arr, null_mat_syn, post)
print(f"\nOmnibus test — Total MI:  obs_mean={omni_mean_tot:.4f} nats  p={omni_pval_tot:.6f}")
print(f"Omnibus test — Synergy:   obs_mean={omni_mean_syn:.4f} nats  p={omni_pval_syn:.6f}")
# ── Null cluster-mass distributions on corrected null ─────────────────────────
ncm_tot = null_cluster_masses(null_mat_tot, null_mean_tot, null_std_tot)
ncm_syn = null_cluster_masses(null_mat_syn, null_mean_syn, null_std_syn)
print(f"\n  Null cluster mass distribution — Total MI:")
print(f"    mean={np.mean(ncm_tot):.4f}  max={np.max(ncm_tot):.4f}  "
      f"95th={np.percentile(ncm_tot,95):.4f}")
print(f"  Null cluster mass distribution — Synergy:")
print(f"    mean={np.mean(ncm_syn):.4f}  max={np.max(ncm_syn):.4f}  "
      f"95th={np.percentile(ncm_syn,95):.4f}")
# ── Cluster-mass tests on corrected null ──────────────────────────────────────
(clo_tot, clp_tot, clpv_tot,
 sig_cluster_tot, _, _) = cluster_mass_test(
    tot_arr, null_mat_tot, null_mean_tot, null_std_tot, ncm_tot)
(clo_syn, clp_syn, clpv_syn,
 sig_cluster_syn, _, _) = cluster_mass_test(
    syn_arr, null_mat_syn, null_mean_syn, null_std_syn, ncm_syn)
print(f"\nCluster-mass — Total MI: {len(clo_tot)} cluster(s)")
for k,(onset,peak,pval) in enumerate(zip(clo_tot,clp_tot,clpv_tot)):
    print(f"  Cluster {k}: onset={t_axis[onset]:.1f}ms  peak={t_axis[peak]:.1f}ms  "
          f"p={pval:.4f}  {'*** SIGNIFICANT' if pval<CLUSTER_ALPHA else 'n.s.'}")
print(f"\nCluster-mass — Synergy: {len(clo_syn)} cluster(s)")
for k,(onset,peak,pval) in enumerate(zip(clo_syn,clp_syn,clpv_syn)):
    print(f"  Cluster {k}: onset={t_axis[onset]:.1f}ms  peak={t_axis[peak]:.1f}ms  "
          f"p={pval:.4f}  {'*** SIGNIFICANT' if pval<CLUSTER_ALPHA else 'n.s.'}")
print(f"\n  Sig bins — Total MI: {sig_cluster_tot.sum()}/{n_tbins}")
print(f"  Sig bins — Synergy:  {sig_cluster_syn.sum()}/{n_tbins}")
# ── Update the 3-source PID CSV with corrected significance values ─────────────
print("\nUpdating table_3source_pid.csv with corrected significance...")
rows=[]
for ti,tv in enumerate(t_axis):
    rows.append({
        'Time_ms':            round(float(tv),1),
        'Total':              round(float(tot_arr[ti]),6),
        'Redundancy':         round(float(red_arr[ti]),6),
        'Synergy':            round(float(syn_arr[ti]),6),
        'UniqueV1':           round(float(uv1_arr[ti]),6),
        'UniqueIT':           round(float(uit_arr[ti]),6),
        'UniqueV4':           round(float(uv4_arr[ti]),6),
        'MI_joint':           round(float(mj_arr[ti]),6),
        'EmergenceIndex':     float('nan') if np.isnan(em_arr[ti])
                              else round(float(em_arr[ti]),6),
        'null_mean_Total':    round(float(null_mean_tot[ti]),6),
        'null_std_Total':     round(float(null_std_tot[ti]),6),
        'null_mean_Syn':      round(float(null_mean_syn[ti]),6),
        'null_std_Syn':       round(float(null_std_syn[ti]),6),
        'sig_cluster_Total':  bool(sig_cluster_tot[ti]),
        'sig_cluster_Syn':    bool(sig_cluster_syn[ti]),
        'omnibus_pval_Total': round(float(omni_pval_tot),6),
        'omnibus_pval_Syn':   round(float(omni_pval_syn),6),
        'omnibus_mean_Total': round(float(omni_mean_tot),6),
        'omnibus_mean_Syn':   round(float(omni_mean_syn),6),
    })
save_csv(pd.DataFrame(rows), 'table_3source_pid.csv', '3-source PID (corrected sig)')
# ── Update cluster stats CSV ───────────────────────────────────────────────────
rows=[]
for k,(onset,peak,pval) in enumerate(zip(clo_tot,clp_tot,clpv_tot)):
    rows.append({'Metric':'TotalMI','Cluster':k,
                 'Onset_ms':round(float(t_axis[onset]),1),
                 'Peak_ms':round(float(t_axis[peak]),1),
                 'Peak_val':round(float(tot_arr[peak]),6),
                 'Cluster_mass':round(float(np.sum(tot_arr[onset:peak+1])),6),
                 'Cluster_pval':round(float(pval),6),
                 'Significant':bool(pval<CLUSTER_ALPHA)})
for k,(onset,peak,pval) in enumerate(zip(clo_syn,clp_syn,clpv_syn)):
    rows.append({'Metric':'Synergy','Cluster':k,
                 'Onset_ms':round(float(t_axis[onset]),1),
                 'Peak_ms':round(float(t_axis[peak]),1),
                 'Peak_val':round(float(syn_arr[peak]),6),
                 'Cluster_mass':round(float(np.sum(syn_arr[onset:peak+1])),6),
                 'Cluster_pval':round(float(pval),6),
                 'Significant':bool(pval<CLUSTER_ALPHA)})
rows.append({'Metric':'TotalMI_omnibus','Cluster':0,
             'Onset_ms':float('nan'),'Peak_ms':float('nan'),
             'Peak_val':round(omni_mean_tot,6),'Cluster_mass':float('nan'),
             'Cluster_pval':round(omni_pval_tot,6),
             'Significant':bool(omni_pval_tot<CLUSTER_ALPHA)})
rows.append({'Metric':'Synergy_omnibus','Cluster':0,
             'Onset_ms':float('nan'),'Peak_ms':float('nan'),
             'Peak_val':round(omni_mean_syn,6),'Cluster_mass':float('nan'),
             'Cluster_pval':round(omni_pval_syn,6),
             'Significant':bool(omni_pval_syn<CLUSTER_ALPHA)})
save_csv(pd.DataFrame(rows), 'table_cluster_stats.csv', 'Cluster stats (corrected)')
print("\nDone. Verify the sanity check above:")
print("  1. null pre-stim mean should be ~0 (confirming scale match)")
print("  2. threshold at t=150ms should be BELOW observed (confirming clusters found)")
print("  3. omnibus p should be << 0.05 (confirming real signal)")
print("  4. at least 1 significant cluster per metric (confirming temporal extent)")'''

Fixing null matrix scale (baseline-subtracting each shuffle column)...
  null_mat_tot pre-stim mean after fix: -0.000000  (should be ~0)
  null_mat_syn pre-stim mean after fix: 0.000000  (should be ~0)
  null_mat_tot post-stim mean: -0.0835 nats
  null_mat_syn post-stim mean: -0.0216 nats
  obs tot_arr  post-stim mean: 0.9855 nats
  obs syn_arr  post-stim mean: 0.1678 nats

  Threshold (null_mean + 2*null_std) at t=150ms:
    Total MI threshold: -0.0376
    Total MI observed:  0.1721
    Synergy  threshold: 0.0960
    Synergy  observed:  0.0000

Omnibus test — Total MI:  obs_mean=0.9855 nats  p=0.003322
Omnibus test — Synergy:   obs_mean=0.1678 nats  p=0.003322

  Null cluster mass distribution — Total MI:
    mean=0.0510  max=0.1517  95th=0.0986
  Null cluster mass distribution — Synergy:
    mean=0.0380  max=0.1790  95th=0.1094

Cluster-mass — Total MI: 2 cluster(s)
  Cluster 0: onset=7.5ms  peak=7.5ms  p=1.0000  n.s.
  Cluster 1: onset=147.5ms  peak=192.5ms  p=0.0033  *** SIGNIFICAN